# AIP304 Coursework 2 — Option 1: CNN image classification

**Student name(s) and ID(s): [TO COMPLETE]**  
**Dataset:** PlantSeg version v7, nine required hosts, host–disease pair as the class.  
**Primary selection metric:** validation macro-F1.

This notebook is the main reproducible entry point. It uses the accompanying Python modules for the complete dataset audit, training, evaluation, plotting, and inference. All reported scores and figures must come from executed code. The source dataset is credited to Wei et al. (2025, 2026); its versioned archive is at https://zenodo.org/records/17719108.

## 本地版：小组运行指南

1. 把完整 PlantSeg v7 数据放在项目的 `plantseg/plantseg/`，其中应有 `Metadata.csv`、`images/` 和 `annotations/`；也可以设置 `PLANTSEG_ROOT`。
2. 从上到下运行。全新结果目录的 `MODE="auto"` 会做完整实验；若已有结果和模型权重则进入演示模式。需要强制重跑时，把配置格改成 `MODE="full"`，程序会使用独立的 `result_local/`。
3. 结果写在 `result/` 或 `result_local/`。本 Notebook 内嵌了人工复核 CSV，单独复制这份 `.ipynb` 时也能重建它；预训练权重仍需联网下载或提前放入结果目录。

## 1. Local setup and execution modes

This notebook reconstructs its Python package source files when they are absent. Run it from the project root or its `code/` directory. Put the complete PlantSeg extraction at `plantseg/plantseg/` under the project root, or set `PLANTSEG_ROOT` to the directory containing `Metadata.csv`. If the official v7 ZIP is present at the project root, the notebook verifies its MD5 and checks the extraction against every ZIP entry. If only extracted files are present, it verifies `Metadata.csv` and all 4,349 task-image hashes against the local copy that was previously checked against the official archive, while reporting that the full archive was not checked in this run.

`MODE="auto"` runs the full audit, training, validation selection, and final test in a fresh result directory. When complete outputs and checkpoints already exist it uses a short demo; if metrics exist without checkpoints it uses recorded-results mode. Set `MODE="full"` to rerun into a separate `result_local/` directory. CUDA is recommended for practical training. The actual environment and elapsed times are saved with the results.

### Complete implementation source

The following expandable cells contain the exact Python source used for data auditing, training, evaluation, plotting, and inference. They write a missing module into the local code/plantseg_cw2 package directory. When separate source files are supplied, their contents must match the embedded copy. This makes the single notebook independently executable after the dataset is available.
When the saved metric files are present but Git-ignored model weights are absent, `MODE="auto"` uses recorded-results mode. Set `MODE="full"` to regenerate into the separate `result_local/` directory.

**阅读提示：**接下来的折叠代码单元把项目模块写入 `code/plantseg_cw2/`。通常只需顺序运行，不必逐行阅读这些长源码；从“数据审计”开始可以按“输入 → 方法 → 输出”理解实验。

**运行提示：**以下都是可执行代码单元；建议使用 Run All，或在单元内按 Shift+Enter。完整源码在本地版中直接展示，方便逐段查阅。

In [ ]:
from pathlib import Path

# Recreate the local package when the notebook is uploaded alone.
working = Path.cwd()
CODE_HOME = working if working.name.lower() == "code" else working / "code"
EMBED_DIR = CODE_HOME / "plantseg_cw2"
(EMBED_DIR / "analysis").mkdir(parents=True, exist_ok=True)
for initializer, content in ((EMBED_DIR / "__init__.py", '"""PlantSeg v7 coursework classification workflow."""\n'),
                             (EMBED_DIR / "analysis" / "__init__.py", '"""Supplementary analysis."""\n')):
    if not initializer.exists():
        initializer.write_text(content, encoding="utf-8")
print("Embedded package directory:", EMBED_DIR)


In [ ]:
# Embedded source: paths.py
module_source = r'''
"""Locate coursework files from source, an installed package, or a notebook."""

from __future__ import annotations

import os
from pathlib import Path


def find_project_root() -> Path:
    configured = os.environ.get("CW2_PROJECT_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not root.is_dir():
            raise FileNotFoundError(f"CW2_PROJECT_ROOT is not a directory: {root}")
        return root
    for start in (Path.cwd().resolve(), Path(__file__).resolve().parent):
        for candidate in (start, *start.parents):
            if (candidate / "code" / "plantseg_cw2" / "dataset.py").is_file():
                return candidate
    raise FileNotFoundError(
        "Cannot locate the coursework2 project root. Run from the repository "
        "or set CW2_PROJECT_ROOT to its absolute path."
    )


PROJECT_ROOT = find_project_root()
'''
module_path = EMBED_DIR / 'paths.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
if module_path.exists():
    if module_path.read_text(encoding='utf-8') != module_source.lstrip('\n'):
        raise ValueError(f'Existing module differs from embedded source: {module_path}')
else:
    module_path.write_text(module_source.lstrip('\n'), encoding='utf-8')
print('Ready:', module_path)


In [ ]:
# Embedded source: dataset.py
module_source = r'''
"""Dataset utilities for the PlantSeg classification coursework."""

from __future__ import annotations

import csv
import json
import os
from pathlib import Path

from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.models import ResNet18_Weights

from .paths import PROJECT_ROOT


HOSTS = (
    "apple", "banana", "citrus", "corn", "potato",
    "soybean", "tomato", "wheat", "zucchini",
)
SPLIT_NAMES = {"Training": "train", "Validation": "val", "Test": "test"}
MANIFEST_COLUMNS = (
    "image_id", "image_path", "host", "disease", "class_name", "class_id",
    "split", "source_index", "annotation_path", "reported_resolution", "metadata_row", "sha256",
    "dhash", "excluded", "exclusion_reason",
)


def dataset_root(path: str | Path | None = None) -> Path:
    """Find the extracted directory that contains Metadata.csv."""
    if path is not None:
        candidates = [Path(path)]
    elif os.environ.get("PLANTSEG_ROOT"):
        candidates = [Path(os.environ["PLANTSEG_ROOT"])]
    else:
        candidates = [
            PROJECT_ROOT / "plantseg" / "plantseg",
            PROJECT_ROOT / "plantseg",
            Path.cwd() / "plantseg" / "plantseg",
            Path.cwd() / "plantseg",
        ]
    for candidate in candidates:
        if (candidate / "Metadata.csv").is_file():
            return candidate.resolve()
        if (candidate / "plantseg" / "Metadata.csv").is_file():
            return (candidate / "plantseg").resolve()
    raise FileNotFoundError(
        "Metadata.csv was not found. Set PLANTSEG_ROOT to the extracted PlantSeg folder."
    )


def result_root(path: str | Path | None = None) -> Path:
    return Path(path or os.environ.get("CW2_RESULT_DIR") or PROJECT_ROOT / "result").resolve()


def write_manifest(rows: list[dict], path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=MANIFEST_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)


def load_manifest(path: str | Path | None = None) -> list[dict]:
    manifest_path = Path(path) if path else result_root() / "manifest.csv"
    with manifest_path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    for row in rows:
        row["class_id"] = int(row["class_id"])
        row["source_index"] = int(row["source_index"])
        row["metadata_row"] = int(row["metadata_row"])
    return rows


def build_manifest(
    data_dir: str | Path | None = None,
    output_dir: str | Path | None = None,
) -> list[dict]:
    """Use only official metadata labels and splits; keep paths relative."""
    source = dataset_root(data_dir)
    destination = result_root(output_dir)
    with (source / "Metadata.csv").open("r", encoding="utf-8-sig", newline="") as handle:
        metadata = [
            (line, row) for line, row in enumerate(csv.DictReader(handle), start=2)
            if row["Plant"].strip().lower() in HOSTS
        ]
    if not metadata:
        raise ValueError("No rows for the nine requested hosts were found in Metadata.csv")

    pairs = sorted({
        (row["Plant"].strip().lower(), row["Disease"].strip().lower())
        for _, row in metadata
    })
    pair_to_id = {pair: index for index, pair in enumerate(pairs)}
    source_indices: dict[tuple[str, str], set[int]] = {pair: set() for pair in pairs}
    rows = []
    seen_names = set()
    for line, item in metadata:
        name = item["Name"].strip()
        if Path(name).name != name:
            raise ValueError(f"Unsafe image name on metadata row {line}: {name}")
        if name in seen_names:
            raise ValueError(f"Duplicate image name in metadata: {name}")
        seen_names.add(name)
        host = item["Plant"].strip().lower()
        disease = item["Disease"].strip().lower()
        pair = (host, disease)
        source_index = int(item["Index"])
        source_indices[pair].add(source_index)
        split = SPLIT_NAMES.get(item["Split"].strip())
        if split is None:
            raise ValueError(f"Unknown official split on metadata row {line}")
        label_name = item["Label file"].strip()
        if Path(label_name).name != label_name:
            raise ValueError(f"Unsafe annotation name on metadata row {line}")
        rows.append({
            "image_id": name,
            "image_path": (Path("images") / split / name).as_posix(),
            "host": host,
            "disease": disease,
            "class_name": f"{host} | {disease}",
            "class_id": pair_to_id[pair],
            "split": split,
            "source_index": source_index,
            "annotation_path": (Path("annotations") / split / label_name).as_posix(),
            "reported_resolution": item["Resolution"].strip(),
            "metadata_row": line,
            "sha256": "",
            "dhash": "",
            "excluded": "0",
            "exclusion_reason": "",
        })
    inconsistent = [pair for pair, indices in source_indices.items() if len(indices) != 1]
    if inconsistent:
        raise ValueError(f"Metadata index varies within classes: {inconsistent}")
    if len({next(iter(v)) for v in source_indices.values()}) != len(pairs):
        raise ValueError("A metadata Index was shared by different host-disease classes")

    mapping = {
        "class_definition": "(host, disease)",
        "class_count": len(pairs),
        "classes": [
            {
                "class_id": index,
                "class_name": f"{host} | {disease}",
                "host": host,
                "disease": disease,
                "source_index": next(iter(source_indices[(host, disease)])),
            }
            for index, (host, disease) in enumerate(pairs)
        ],
    }
    destination.mkdir(parents=True, exist_ok=True)
    (destination / "class_mapping.json").write_text(
        json.dumps(mapping, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    write_manifest(rows, destination / "manifest.csv")
    return rows


def is_included(row: dict) -> bool:
    return str(row.get("excluded", "0")).strip().lower() not in {"1", "true", "yes"}


def make_transforms(augment: bool = False):
    """Use the ResNet-18 ImageNet normalization for every split."""
    norm = transforms.Normalize(
        mean=ResNet18_Weights.IMAGENET1K_V1.transforms().mean,
        std=ResNet18_Weights.IMAGENET1K_V1.transforms().std,
    )
    # Match the deterministic preprocessing specified by the pretrained weights.
    steps = [transforms.Resize(256), transforms.CenterCrop(224)]
    if augment:
        steps.extend([
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
        ])
    steps.extend([transforms.ToTensor(), norm])
    return transforms.Compose(steps)


class PlantDiseaseDataset(Dataset):
    def __init__(
        self,
        manifest: str | Path | list[dict],
        data_dir: str | Path | None,
        split: str,
        transform=None,
    ):
        if split not in {"train", "val", "test"}:
            raise ValueError(f"Unknown split: {split}")
        all_rows = load_manifest(manifest) if isinstance(manifest, (str, Path)) else manifest
        self.rows = [row for row in all_rows if row["split"] == split and is_included(row)]
        self.data_dir = dataset_root(data_dir)
        self.transform = transform or make_transforms(False)

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int):
        row = self.rows[index]
        with Image.open(self.data_dir / row["image_path"]) as image:
            rgb = image.convert("RGB")
        return self.transform(rgb), int(row["class_id"])
'''
module_path = EMBED_DIR / 'dataset.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
if module_path.exists():
    if module_path.read_text(encoding='utf-8') != module_source.lstrip('\n'):
        raise ValueError(f'Existing module differs from embedded source: {module_path}')
else:
    module_path.write_text(module_source.lstrip('\n'), encoding='utf-8')
print('Ready:', module_path)


In [ ]:
# Embedded source: audit_dataset.py
module_source = r'''
"""Build the fixed classification manifest and audit its source images."""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import zlib
import zipfile
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath

from PIL import Image

from .dataset import HOSTS, build_manifest, dataset_root, is_included, result_root, write_manifest
from .paths import PROJECT_ROOT


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def md5_file(path: Path) -> str:
    digest = hashlib.md5()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def difference_hash(image: Image.Image) -> int:
    """Return a 64-bit dHash; low Hamming distance suggests visual similarity."""
    pixels = image.convert("L").resize((9, 8), Image.Resampling.LANCZOS).tobytes()
    bits = 0
    for row in range(8):
        for col in range(8):
            bits = (bits << 1) | int(pixels[row * 9 + col] > pixels[row * 9 + col + 1])
    return bits


def inspect_images(rows: list[dict], source: Path) -> tuple[list[dict], list[dict]]:
    image_errors = []
    resolution_mismatches = []
    for row in rows:
        path = source / row["image_path"]
        if not path.is_file():
            row["excluded"] = "1"
            row["exclusion_reason"] = "missing_image"
            image_errors.append({"image_id": row["image_id"], "problem": "missing"})
            continue
        try:
            row["sha256"] = sha256_file(path)
            with Image.open(path) as image:
                image.load()
                width, height = image.size
                row["dhash"] = f"{difference_hash(image):016x}"
                if width < 1 or height < 1:
                    raise ValueError("invalid image dimensions")
                if row["reported_resolution"] != f"{width}x{height}":
                    resolution_mismatches.append({
                        "image_id": row["image_id"],
                        "metadata": row["reported_resolution"],
                        "actual": f"{width}x{height}",
                    })
        except (OSError, ValueError) as error:
            row["excluded"] = "1"
            row["exclusion_reason"] = "corrupt_image"
            image_errors.append({"image_id": row["image_id"], "problem": str(error)})
    return image_errors, resolution_mismatches


def exclude_exact_duplicates(rows: list[dict]) -> list[dict]:
    """Filter a temporary copy for near-duplicate candidate discovery only."""
    groups = defaultdict(list)
    for row in rows:
        if is_included(row) and row["sha256"]:
            groups[row["sha256"]].append(row)
    split_priority = {"train": 0, "val": 1, "test": 2}
    duplicates = []
    for digest, group in groups.items():
        if len(group) < 2:
            continue
        ordered = sorted(group, key=lambda row: (
            split_priority[row["split"]], row["image_id"]
        ))
        best_split = ordered[0]["split"]
        retained = [row for row in ordered if row["split"] == best_split]
        for row in ordered:
            if row["split"] != best_split:
                row["excluded"] = "1"
                row["exclusion_reason"] = f"cross_split_exact_duplicate_of:{retained[0]['image_id']}"
        duplicates.append({
            "sha256": digest,
            "retained": [row["image_id"] for row in retained],
            "members": [
                {"image_id": row["image_id"], "split": row["split"],
                 "class_name": row["class_name"]}
                for row in ordered
            ],
            "cross_split": len({row["split"] for row in ordered}) > 1,
            "conflicting_labels": len({row["class_id"] for row in ordered}) > 1,
        })
    return duplicates


def find_near_duplicates(rows: list[dict], output: Path, threshold: int = 4) -> list[dict]:
    """Record cross-split candidates only; do not delete by hash threshold."""
    by_split = {
        split: [(row, int(row["dhash"], 16)) for row in rows
                if row["split"] == split and is_included(row) and row["dhash"]]
        for split in ("train", "val", "test")
    }
    candidates = []
    for first, second in (("train", "val"), ("train", "test"), ("val", "test")):
        for row_a, hash_a in by_split[first]:
            for row_b, hash_b in by_split[second]:
                distance = (hash_a ^ hash_b).bit_count()
                if distance <= threshold:
                    candidates.append({
                        "image_a": row_a["image_id"], "split_a": first,
                        "class_a": row_a["class_name"],
                        "image_b": row_b["image_id"], "split_b": second,
                        "class_b": row_b["class_name"],
                        "hamming_distance": distance,
                    })
    candidates.sort(key=lambda item: (
        item["hamming_distance"], item["image_a"], item["image_b"]
    ))
    with (output / "tables" / "near_duplicate_candidates.csv").open(
        "w", encoding="utf-8", newline=""
    ) as handle:
        columns = (
            "image_a", "split_a", "class_a", "image_b", "split_b", "class_b",
            "hamming_distance",
        )
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(candidates)
    return candidates


def apply_reviewed_near_duplicates(
    rows: list[dict], candidates: list[dict], output: Path
) -> dict:
    """Replay visual decisions to document overlaps, without altering model rows."""
    review_path = output / "tables" / "near_duplicate_review.csv"
    if not review_path.is_file():
        for filename in ("near_duplicate_overlap_risk.csv", "near_duplicate_exclusions.csv"):
            with (output / "tables" / filename).open("w", encoding="utf-8", newline="") as handle:
                csv.writer(handle).writerow(("image_id", "split", "matching_image_id", "matching_split"))
        return {"reviewed_pairs": 0, "confirmed_pairs": 0,
                "uncertain_pairs": 0, "excluded_images": [],
                "potential_overlap_images": [],
                "review_sha256": None, "dhash_reviewed_pairs": 0,
                "dhash_confirmed_pairs": 0, "phash_extra_pairs": 0,
                "phash_extra_confirmed_pairs": 0}
    with review_path.open("r", encoding="utf-8", newline="") as handle:
        decisions = list(csv.DictReader(handle))
    expected = {(item["image_a"], item["image_b"]) for item in candidates}
    observed = {(item["image_a"], item["image_b"]) for item in decisions}
    if not expected.issubset(observed) or len(observed) != len(decisions):
        raise ValueError(
            "Near-duplicate review must cover every current dHash candidate exactly once."
        )
    allowed = {"same_photo", "different_photo", "uncertain"}
    if any(item["decision"] not in allowed for item in decisions):
        raise ValueError("Review decisions must be same_photo, different_photo, or uncertain")

    lookup = {row["image_id"]: row for row in rows if is_included(row)}
    for item in decisions:
        for suffix in ("a", "b"):
            image_id = item[f"image_{suffix}"]
            if image_id not in lookup:
                raise ValueError(f"Reviewed image is absent from audited images: {image_id}")
            source_row = lookup[image_id]
            if (item[f"split_{suffix}"] != source_row["split"]
                    or item[f"class_{suffix}"] != source_row["class_name"]
                    or item[f"sha256_{suffix}"] != source_row["sha256"]):
                raise ValueError(f"Reviewed image changed since visual review: {image_id}")
        if item["split_a"] == item["split_b"]:
            raise ValueError("Near-duplicate review must contain cross-split pairs only")

    parent = {}

    def find(key):
        parent.setdefault(key, key)
        while parent[key] != key:
            parent[key] = parent[parent[key]]
            key = parent[key]
        return key

    def union(first, second):
        parent[find(first)] = find(second)

    confirmed = [item for item in decisions if item["decision"] == "same_photo"]
    for item in confirmed:
        union(item["image_a"], item["image_b"])
    groups = defaultdict(list)
    for image_id in parent:
        groups[find(image_id)].append(lookup[image_id])
    split_priority = {"train": 0, "val": 1, "test": 2}
    potential_overlap = []
    for group in groups.values():
        best_split = min((row["split"] for row in group), key=split_priority.get)
        reference_id = min(row["image_id"] for row in group if row["split"] == best_split)
        for row in group:
            if row["split"] != best_split:
                potential_overlap.append({
                    "image_id": row["image_id"], "split": row["split"],
                    "matching_image_id": reference_id, "matching_split": best_split,
                })
    with (output / "tables" / "near_duplicate_overlap_risk.csv").open(
        "w", encoding="utf-8", newline=""
    ) as handle:
        writer = csv.DictWriter(
            handle, fieldnames=("image_id", "split", "matching_image_id", "matching_split")
        )
        writer.writeheader()
        writer.writerows(potential_overlap)
    # Overwrite the previous run's exclusion table to avoid stale claims.
    with (output / "tables" / "near_duplicate_exclusions.csv").open(
        "w", encoding="utf-8", newline=""
    ) as handle:
        writer = csv.DictWriter(
            handle, fieldnames=("image_id", "split", "retained_image_id", "retained_split")
        )
        writer.writeheader()
    return {
        "reviewed_pairs": len(decisions),
        "confirmed_pairs": len(confirmed),
        "uncertain_pairs": sum(item["decision"] == "uncertain" for item in decisions),
        "review_sha256": sha256_file(review_path),
        "dhash_reviewed_pairs": sum(item["candidate_source"] == "dhash" for item in decisions),
        "dhash_confirmed_pairs": sum(item["candidate_source"] == "dhash"
                                      and item["decision"] == "same_photo" for item in decisions),
        "phash_extra_pairs": sum(item["candidate_source"] == "phash" for item in decisions),
        "phash_extra_confirmed_pairs": sum(item["candidate_source"] == "phash"
                                           and item["decision"] == "same_photo" for item in decisions),
        "excluded_images": [],
        "potential_overlap_images": potential_overlap,
    }


def count_by_class(rows: list[dict], mapping: dict, output: Path) -> list[dict]:
    raw = Counter((int(row["class_id"]), row["split"]) for row in rows)
    used = Counter((int(row["class_id"]), row["split"]) for row in rows if is_included(row))
    counts = []
    for item in mapping["classes"]:
        class_id = item["class_id"]
        record = {key: item[key] for key in ("class_id", "class_name", "host", "disease")}
        for split in ("train", "val", "test"):
            record[f"raw_{split}"] = raw[class_id, split]
            record[f"used_{split}"] = used[class_id, split]
        record["raw_total"] = sum(record[f"raw_{split}"] for split in ("train", "val", "test"))
        record["used_total"] = sum(record[f"used_{split}"] for split in ("train", "val", "test"))
        counts.append(record)
    with (output / "tables" / "class_counts.csv").open(
        "w", encoding="utf-8", newline=""
    ) as handle:
        writer = csv.DictWriter(handle, fieldnames=list(counts[0]))
        writer.writeheader()
        writer.writerows(counts)
    return counts


def plot_distribution(counts: list[dict], output: Path) -> None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    names = [item["class_name"] for item in counts]
    indices = list(range(len(counts)))
    fig, ax = plt.subplots(figsize=(11, 15))
    left = [0] * len(counts)
    for split, color in (("train", "#4C78A8"), ("val", "#F2CF5B"), ("test", "#E07B54")):
        values = [item[f"used_{split}"] for item in counts]
        ax.barh(indices, values, left=left, color=color, label=split, height=0.8)
        left = [a + b for a, b in zip(left, values)]
    ax.set_yticks(indices, names, fontsize=7.5)
    ax.invert_yaxis()
    ax.set_xlabel("Images in the supplied official splits")
    ax.set_title("PlantSeg nine-host class distribution")
    ax.legend(loc="lower right")
    ax.grid(axis="x", alpha=0.2)
    ax.set_axisbelow(True)
    fig.tight_layout()
    fig.subplots_adjust(left=0.35, right=0.98)
    fig.savefig(output / "figures" / "class_distribution.png", dpi=180)
    plt.close(fig)


def plot_examples(rows: list[dict], source: Path, output: Path) -> None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    examples = []
    for host in HOSTS:
        candidate = next((row for row in rows if row["host"] == host
                          and row["split"] == "train" and is_included(row)), None)
        if candidate:
            examples.append(candidate)
    fig, axes = plt.subplots(3, 3, figsize=(11, 9))
    for axis, row in zip(axes.flat, examples):
        with Image.open(source / row["image_path"]) as image:
            axis.imshow(image.convert("RGB"))
        axis.set_title(row["class_name"], fontsize=8)
        axis.axis("off")
    for axis in list(axes.flat)[len(examples):]:
        axis.axis("off")
    fig.suptitle("Representative training images (one per host)", fontsize=12)
    fig.tight_layout()
    fig.savefig(output / "figures" / "training_examples.png", dpi=150)
    plt.close(fig)


def plot_near_duplicate_preview(
    candidates: list[dict], rows: list[dict], source: Path, output: Path,
    limit: int = 10, filename: str = "near_duplicate_preview.png",
) -> None:
    """Make a small review sheet, keeping the candidate status explicit."""
    if not candidates:
        return
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    lookup = {row["image_id"]: row for row in rows}
    selected = candidates[:limit]
    fig, axes = plt.subplots(len(selected), 2, figsize=(10, 3 * len(selected)))
    if len(selected) == 1:
        axes = [axes]
    for pair, (left, right) in zip(selected, axes):
        for axis, key, split_key, class_key in (
            (left, "image_a", "split_a", "class_a"),
            (right, "image_b", "split_b", "class_b"),
        ):
            with Image.open(source / lookup[pair[key]]["image_path"]) as image:
                axis.imshow(image.convert("RGB"))
            axis.set_title(
                f"{pair[split_key]}: {pair[key]}\n{pair[class_key]}", fontsize=7
            )
            axis.axis("off")
        left.text(
            0, -0.05,
            f"candidate {pair.get('_review_index', '?')}; "
            f"dHash distance = {pair['hamming_distance']}; visual review required",
            transform=left.transAxes, fontsize=8,
        )
    fig.suptitle("Cross-split near-duplicate candidates (not automatically excluded)")
    fig.tight_layout(rect=(0, 0, 1, 0.99))
    fig.savefig(output / "figures" / filename, dpi=130)
    plt.close(fig)


def source_summary(source: Path) -> dict:
    counts = {}
    total_bytes = 0
    total_files = 0
    for kind in ("images", "annotations"):
        counts[kind] = {}
        for split in ("train", "val", "test"):
            files = [path for path in (source / kind / split).iterdir() if path.is_file()]
            counts[kind][split] = len(files)
            total_files += len(files)
            total_bytes += sum(path.stat().st_size for path in files)
    for path in source.iterdir():
        if path.is_file():
            total_files += 1
            total_bytes += path.stat().st_size
    counts["all_files"] = total_files
    counts["all_bytes"] = total_bytes
    with (source / "Metadata.csv").open("r", encoding="utf-8-sig", newline="") as handle:
        metadata = list(csv.DictReader(handle))
    counts["metadata_rows"] = len(metadata)
    counts["metadata_missing_images"] = sum(
        not (source / "images" / {
            "Training": "train", "Validation": "val", "Test": "test"
        }[row["Split"]] / row["Name"]).is_file()
        for row in metadata
    )
    return counts


def verify_archive_extraction(archive: Path, source: Path) -> dict:
    """Check every extracted file against the versioned ZIP's size and CRC."""
    missing = []
    size_mismatches = []
    crc_mismatches = []
    archive_paths = set()
    with zipfile.ZipFile(archive) as bundle:
        members = [item for item in bundle.infolist() if not item.is_dir()]
        for item in members:
            parts = PurePosixPath(item.filename).parts
            if not parts or parts[0] != "plantseg" or ".." in parts:
                raise ValueError(f"Unexpected or unsafe ZIP member: {item.filename}")
            relative = Path(*parts[1:])
            archive_paths.add(relative.as_posix())
            extracted = source / relative
            if not extracted.is_file():
                missing.append(item.filename)
                continue
            if extracted.stat().st_size != item.file_size:
                size_mismatches.append(item.filename)
                continue
            checksum = 0
            with extracted.open("rb") as handle:
                for block in iter(lambda: handle.read(1024 * 1024), b""):
                    checksum = zlib.crc32(block, checksum)
            if checksum != item.CRC:
                crc_mismatches.append(item.filename)
    extracted_paths = {path.relative_to(source).as_posix()
                       for path in source.rglob("*") if path.is_file()}
    extra = sorted(extracted_paths - archive_paths)
    return {
        "archive_file_count": len(members),
        "archive_uncompressed_bytes": sum(item.file_size for item in members),
        "missing_count": len(missing), "size_mismatch_count": len(size_mismatches),
        "crc_mismatch_count": len(crc_mismatches), "extra_extracted_count": len(extra),
        "missing_examples": missing[:10], "size_mismatch_examples": size_mismatches[:10],
        "crc_mismatch_examples": crc_mismatches[:10], "extra_extracted_examples": extra[:10],
        "matches_extraction": not (missing or size_mismatches or crc_mismatches or extra),
    }


def archive_extraction_check(archive: Path, source: Path, output: Path,
                             archive_md5: str) -> dict:
    """Reuse a fresh full-file verification; otherwise verify every file now."""
    cached_path = output / "source_archive_verification.json"
    if cached_path.is_file() and cached_path.stat().st_mtime >= archive.stat().st_mtime:
        cached = json.loads(cached_path.read_text(encoding="utf-8"))
        if (cached.get("archive") == str(archive.resolve())
                and cached.get("archive_md5") == archive_md5
                and cached.get("extraction_matches_archive") is True):
            latest_extracted_change = max(
                (path.stat().st_mtime for path in source.rglob("*") if path.is_file()),
                default=0,
            )
            if cached_path.stat().st_mtime >= latest_extracted_change:
                return {
                    "archive_file_count": cached["zip_file_count"],
                    "archive_uncompressed_bytes": cached["zip_uncompressed_bytes"],
                    "missing_count": len(cached["missing_files"]),
                    "size_mismatch_count": len(cached["size_mismatch"]),
                    "crc_mismatch_count": len(cached["crc_mismatch"]),
                    "extra_extracted_count": len(cached["extra_files"]),
                    "matches_extraction": True,
                    "verification_source": "full size/CRC scan in source_archive_verification.json",
                }
    result = verify_archive_extraction(archive, source)
    result["verification_source"] = "full size/CRC scan by audit_dataset.py"
    return result


def write_split_stats(summary: dict, output: Path) -> None:
    path = output / "tables" / "split_stats.csv"
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=(
            "split", "raw_images", "used_images", "excluded_images", "class_count"
        ))
        writer.writeheader()
        for split, counts in summary["split_counts"].items():
            writer.writerow({
                "split": split,
                "raw_images": counts["raw"],
                "used_images": counts["used"],
                "excluded_images": counts["raw"] - counts["used"],
                "class_count": summary["classes"],
            })
        writer.writerow({
            "split": "total",
            "raw_images": summary["raw_images"],
            "used_images": summary["used_images"],
            "excluded_images": summary["raw_images"] - summary["used_images"],
            "class_count": summary["classes"],
        })


def audit_dataset(
    data_dir: str | Path | None = None,
    output_dir: str | Path | None = None,
    near_threshold: int = 4,
) -> dict:
    source = dataset_root(data_dir)
    output = result_root(output_dir)
    (output / "figures").mkdir(parents=True, exist_ok=True)
    (output / "tables").mkdir(parents=True, exist_ok=True)
    rows = build_manifest(source, output)
    mapping = json.loads((output / "class_mapping.json").read_text(encoding="utf-8"))
    image_errors, resolution_mismatches = inspect_images(rows, source)
    missing_masks = [row["image_id"] for row in rows
                     if not (source / row["annotation_path"]).is_file()]
    # The model uses every readable source row in its supplied split. Make a
    # separate copy solely to reproduce the earlier near-duplicate review set;
    # exact copies are omitted from candidate discovery, not model training.
    review_rows = [dict(row) for row in rows]
    duplicates = exclude_exact_duplicates(review_rows)
    for group in duplicates:
        group["retained_in_model"] = [member["image_id"] for member in group["members"]]
        group.pop("retained", None)
    candidates = find_near_duplicates(review_rows, output, near_threshold)
    review = apply_reviewed_near_duplicates(review_rows, candidates, output)
    write_manifest(rows, output / "manifest.csv")
    counts = count_by_class(rows, mapping, output)
    plot_distribution(counts, output)
    plot_examples(rows, source, output)
    plot_near_duplicate_preview(candidates, rows, source, output)
    used = [row for row in rows if is_included(row)]
    split_counts = {
        split: {"raw": sum(row["split"] == split for row in rows),
                "used": sum(row["split"] == split for row in used)}
        for split in ("train", "val", "test")
    }
    train_counts = [item["used_train"] for item in counts]
    info = json.loads((source / "annotation_train.json").read_text(encoding="utf-8"))["info"]
    archive = next((path for path in (PROJECT_ROOT / "plantseg.zip",
               source.parent / "plantseg.zip") if path.is_file()), None)
    archive_md5 = md5_file(archive) if archive else None
    official_md5 = "9358a66dff88cdd15c4fe009763c40a3"
    extraction_check = archive_extraction_check(archive, source, output, archive_md5) if archive else None
    release_verified = archive_md5 == official_md5 and bool(extraction_check and extraction_check["matches_extraction"])
    summary = {
        "dataset_release": "PlantSeg v7 verified" if release_verified else "unverified",
        "version_note": (
            "Local plantseg.zip matches the official v7 MD5; every extracted file matches the archive's size and CRC."
            if release_verified else
            "The supplied extraction could not be verified against the PlantSeg v7 archive. "
            "COCO annotation info.version is %s and does not establish the PlantSeg release."
            % info.get("version", "missing")
        ),
        "official_v7_record": "https://zenodo.org/records/17719108",
        "official_v7_archive_md5": official_md5,
        "local_archive_present": archive is not None,
        "local_archive_bytes": archive.stat().st_size if archive else None,
        "local_archive_md5": archive_md5,
        "archive_extraction_check": extraction_check,
        "source": source_summary(source),
        "nine_hosts": list(HOSTS),
        "raw_images": len(rows),
        "used_images": len(used),
        "classes": len(counts),
        "split_counts": split_counts,
        "missing_or_corrupt_images": image_errors,
        "missing_masks": missing_masks,
        "resolution_mismatches": resolution_mismatches,
        "exact_duplicate_groups": duplicates,
        "exact_duplicate_group_count": len(duplicates),
        "cross_split_exact_duplicate_group_count": sum(item["cross_split"] for item in duplicates),
        "cross_split_conflicting_label_group_count": sum(
            item["cross_split"] and item["conflicting_labels"] for item in duplicates
        ),
        "within_split_conflicting_label_group_count": sum(
            not item["cross_split"] and item["conflicting_labels"] for item in duplicates
        ),
        "near_duplicate_threshold": near_threshold,
        "cross_split_near_duplicate_candidate_count": len(candidates),
        "near_duplicate_reviewed_pair_count": review["reviewed_pairs"],
        "near_duplicate_confirmed_pair_count": review["confirmed_pairs"],
        "near_duplicate_uncertain_pair_count": review["uncertain_pairs"],
        "near_duplicate_review_sha256": review["review_sha256"],
        "dhash_reviewed_pair_count": review["dhash_reviewed_pairs"],
        "dhash_confirmed_pair_count": review["dhash_confirmed_pairs"],
        "phash_extra_reviewed_pair_count": review["phash_extra_pairs"],
        "phash_extra_confirmed_pair_count": review["phash_extra_confirmed_pairs"],
        "near_duplicate_exclusions": review["excluded_images"],
        "near_duplicate_overlap_risk_images": review.get("potential_overlap_images", []),
        "near_duplicate_note": (
            "64-bit dHash and pHash generated candidate pairs; visual decisions were recorded. "
            "All readable images remain in their original splits, including confirmed cross-split copies. "
            "Validation and test scores may therefore be optimistic."
        ),
        "train_class_min": min(train_counts),
        "train_class_max": max(train_counts),
        "train_imbalance_ratio": max(train_counts) / min(train_counts) if min(train_counts) else None,
        "exclusion_rule": (
            "Use every readable image in its supplied train, validation, or test split. "
            "No duplicate-based exclusions or split reassignments are applied. "
            "This preserves the supplied split assignments but retains same-photo overlap across partitions."
        ),
        "duplicate_policy": "original_splits_no_duplicate_exclusions",
    }
    (output / "dataset_audit.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    write_split_stats(summary, output)
    lines = [
        "# Dataset audit summary", "",
        f"Selected nine-host images: {len(rows):,} raw; {len(used):,} used in original splits.",
        f"Host-disease classes: {len(counts)}.",
        "Official split counts (raw → used): " + ", ".join(
            f"{split} {data['raw']:,} → {data['used']:,}"
            for split, data in split_counts.items()
        ) + ".",
        f"Missing/corrupt selected images: {len(image_errors)}; missing masks: {len(missing_masks)}.",
        f"Exact duplicate groups: {len(duplicates)}; cross-split groups: "
        f"{summary['cross_split_exact_duplicate_group_count']}.",
        f"Groups with conflicting class labels: "
        f"{summary['cross_split_conflicting_label_group_count']} cross-split and "
        f"{summary['within_split_conflicting_label_group_count']} within one split; "
        "all copies remain in their supplied splits.",
        f"Cross-split dHash candidates (distance ≤ {near_threshold}): {len(candidates)}; "
        f"additional pHash candidates: {review['phash_extra_pairs']}; "
        f"visually confirmed pairs: {review['confirmed_pairs']}; "
        "all copies retained at the user's direction.",
        f"Train class size range: {min(train_counts)}–{max(train_counts)} "
        f"(ratio {summary['train_imbalance_ratio']:.2f}:1).",
        "", "## Version", "", summary["version_note"], "",
        "## Leakage rule", "", summary["exclusion_rule"], "",
    ]
    (output / "tables" / "dataset_audit_summary.md").write_text(
        "\n".join(lines), encoding="utf-8"
    )
    return summary


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-dir", type=Path, default=None)
    parser.add_argument("--output-dir", type=Path, default=None)
    parser.add_argument("--near-threshold", type=int, default=4)
    options = parser.parse_args()
    audit = audit_dataset(options.data_dir, options.output_dir, options.near_threshold)
    print(json.dumps({
        "raw_images": audit["raw_images"],
        "used_images": audit["used_images"],
        "classes": audit["classes"],
        "split_counts": audit["split_counts"],
        "exact_duplicate_groups": audit["exact_duplicate_group_count"],
        "cross_split_near_duplicate_candidates": audit["cross_split_near_duplicate_candidate_count"],
    }, indent=2))
'''
module_path = EMBED_DIR / 'audit_dataset.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
if module_path.exists():
    if module_path.read_text(encoding='utf-8') != module_source.lstrip('\n'):
        raise ValueError(f'Existing module differs from embedded source: {module_path}')
else:
    module_path.write_text(module_source.lstrip('\n'), encoding='utf-8')
print('Ready:', module_path)


In [ ]:
# Embedded source: evaluate.py
module_source = r'''
"""Metrics and figures for the fixed PlantSeg classification protocol."""

from __future__ import annotations

import csv
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from .dataset import PlantDiseaseDataset, dataset_root, load_manifest, make_transforms, result_root


def scores(y_true, y_pred, class_count: int) -> dict:
    """Compute per-class scores over the same fixed class list in every run."""
    true = np.asarray(y_true, dtype=np.int64)
    pred = np.asarray(y_pred, dtype=np.int64)
    if len(true) != len(pred) or len(true) == 0:
        raise ValueError("Evaluation needs matching nonempty labels and predictions")
    if (true < 0).any() or (true >= class_count).any() or (pred < 0).any() or (pred >= class_count).any():
        raise ValueError("Class id outside the fixed class mapping")
    matrix = np.bincount(true * class_count + pred, minlength=class_count**2).reshape(class_count, class_count)
    support = matrix.sum(axis=1)
    predicted = matrix.sum(axis=0)
    correct = matrix.diagonal()
    precision = np.divide(correct, predicted, out=np.zeros(class_count, dtype=float), where=predicted != 0)
    recall = np.divide(correct, support, out=np.zeros(class_count, dtype=float), where=support != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(class_count, dtype=float), where=(precision + recall) != 0)
    return {
        "accuracy": float(correct.sum() / len(true)),
        "balanced_accuracy": float(recall.mean()),
        "macro_f1": float(f1.mean()),
        "per_class": [
            {"class_id": i, "precision": float(precision[i]), "recall": float(recall[i]),
             "f1": float(f1[i]), "support": int(support[i])}
            for i in range(class_count)
        ],
        "confusion_matrix": matrix.tolist(),
        "class_count": class_count,
        "undefined_metric_rule": "Precision, recall and F1 are 0 when their denominator is 0; all mapped classes are included in macro averages.",
    }


def majority_prediction(manifest: list[dict], split: str, class_count: int):
    from .dataset import is_included
    training = [r for r in manifest if r["split"] == "train" and is_included(r)]
    target = [r for r in manifest if r["split"] == split and is_included(r)]
    counts = np.bincount([r["class_id"] for r in training], minlength=class_count)
    majority_id = int(np.argmax(counts))
    result = scores([r["class_id"] for r in target], [majority_id] * len(target), class_count)
    result["majority_class_id"] = majority_id
    return result


def predict_model(model, split: str, manifest: list[dict], data_dir: Path, class_count: int,
                  device: torch.device, batch_size: int = 64) -> tuple[dict, list[dict]]:
    ds = PlantDiseaseDataset(manifest, data_dir, split, make_transforms(False))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0,
                        pin_memory=device.type == "cuda")
    y_true, y_pred, confidence = [], [], []
    model.eval()
    feature_model = torch.nn.Sequential(*list(model.children())[:-1]).eval()
    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", enabled=device.type == "cuda"):
                features = feature_model(images)
            logits = model.fc(features.flatten(1).float())
            probabilities = torch.softmax(logits.float(), dim=1)
            conf, prediction = probabilities.max(dim=1)
            y_true.extend(labels.numpy().tolist())
            y_pred.extend(prediction.cpu().numpy().tolist())
            confidence.extend(conf.cpu().numpy().tolist())
    result = scores(y_true, y_pred, class_count)
    predictions = [
        {"image_id": row["image_id"], "image_path": row["image_path"],
         "true_class": int(row["class_id"]), "pred_class": int(pred),
         "confidence": float(conf)}
        for row, pred, conf in zip(ds.rows, y_pred, confidence)
    ]
    return result, predictions


def write_rows(path: Path, rows: list[dict], columns: list[str] | None = None) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows and not columns:
        return
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns or list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def plot_training_curves(output: Path, experiments: list[dict]) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.7))
    for entry in experiments:
        path = output / "logs" / f"{entry['experiment']}_history.csv"
        if not path.exists():
            continue
        with path.open(encoding="utf-8", newline="") as handle:
            rows = list(csv.DictReader(handle))
        epoch = [int(r["epoch"]) for r in rows]
        axes[0].plot(epoch, [float(r["train_loss"]) for r in rows], label=entry["experiment"])
        axes[1].plot(epoch, [float(r["val_loss"]) for r in rows], label=entry["experiment"])
        axes[2].plot(epoch, [float(r["val_macro_f1"]) for r in rows], label=entry["experiment"])
        best = int(entry["best_epoch"])
        axes[2].scatter([best], [float(rows[best - 1]["val_macro_f1"])], s=24)
    for ax, title, ylabel in zip(axes, ["Training loss", "Validation loss", "Validation macro-F1"],
                                 ["Cross-entropy", "Cross-entropy", "Macro-F1"]):
        ax.set(title=title, xlabel="Epoch", ylabel=ylabel)
        ax.grid(alpha=0.25)
    axes[2].legend(fontsize=7, loc="lower right")
    fig.tight_layout()
    fig.savefig(output / "figures" / "training_curves.png", dpi=180)
    plt.close(fig)


def plot_model_comparison(output: Path, summary: list[dict], test: dict | None = None) -> None:
    rows = summary
    names = [r["experiment"].replace("_", " ") for r in rows]
    x = np.arange(len(rows))
    width = 0.25
    fig, ax = plt.subplots(figsize=(9, 4.1))
    for offset, key, label in [(-width, "val_macro_f1", "Macro-F1"), (0, "val_accuracy", "Accuracy"),
                               (width, "val_balanced_accuracy", "Balanced accuracy")]:
        ax.bar(x + offset, [float(r[key]) for r in rows], width, label=label)
    ax.set(xticks=x, xticklabels=names, ylim=(0, 1), ylabel="Validation score",
           title="Controlled experiment comparison")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    fig.savefig(output / "figures" / "model_comparison.png", dpi=180)
    plt.close(fig)


def plot_confusion(output: Path, metrics: dict) -> None:
    matrix = np.asarray(metrics["confusion_matrix"])
    fig, ax = plt.subplots(figsize=(10, 9))
    im = ax.imshow(matrix, cmap="Blues", interpolation="nearest")
    fig.colorbar(im, ax=ax, fraction=0.035)
    ax.set(title="Selected model: test confusion matrix", xlabel="Predicted class ID", ylabel="True class ID")
    ax.set_xticks(np.arange(len(matrix)))
    ax.set_yticks(np.arange(len(matrix)))
    ax.tick_params(labelsize=5)
    fig.tight_layout()
    fig.savefig(output / "figures" / "confusion_matrix.png", dpi=180)
    plt.close(fig)


def plot_examples(output: Path, predictions: list[dict], mapping: dict, data_dir: Path) -> None:
    from PIL import Image
    correct = [r for r in predictions if r["true_class"] == r["pred_class"]]
    incorrect = [r for r in predictions if r["true_class"] != r["pred_class"]]
    chosen = correct[:3] + sorted(incorrect, key=lambda r: -r["confidence"])[:3]
    if not chosen:
        return
    names = [x["class_name"] for x in mapping["classes"]]
    fig, axes = plt.subplots(2, 3, figsize=(11.5, 7.3))
    for ax, row in zip(axes.flat, chosen):
        with Image.open(data_dir / row["image_path"]) as image:
            ax.imshow(image.convert("RGB"))
        ax.axis("off")
        ax.set_title(f"True: {names[row['true_class']]}\nPred: {names[row['pred_class']]} ({row['confidence']:.2f})", fontsize=8)
    for ax in list(axes.flat)[len(chosen):]:
        ax.axis("off")
    fig.suptitle("Test examples (softmax score is not calibrated probability)", fontsize=11)
    fig.tight_layout()
    fig.savefig(output / "figures" / "prediction_examples.png", dpi=180)
    plt.close(fig)


def make_final_plots(output_dir: str | Path | None = None) -> None:
    output = result_root(output_dir)
    with (output / "experiment_summary.csv").open(encoding="utf-8", newline="") as handle:
        summary = list(csv.DictReader(handle))
    plot_training_curves(output, summary)
    plot_model_comparison(output, summary)
    test_path = output / "final_test_metrics.json"
    if test_path.exists():
        test = json.loads(test_path.read_text(encoding="utf-8"))
        chosen = test["selected_model"]
        plot_confusion(output, test["experiments"][chosen])
        mapping = json.loads((output / "class_mapping.json").read_text(encoding="utf-8"))
        pred_path = output / "tables" / "test_predictions.csv"
        if pred_path.exists():
            with pred_path.open(encoding="utf-8", newline="") as handle:
                rows = list(csv.DictReader(handle))
            for row in rows:
                row["true_class"] = int(row["true_class"])
                row["pred_class"] = int(row["pred_class"])
                row["confidence"] = float(row["confidence"])
            plot_examples(output, rows, mapping, dataset_root())
'''
module_path = EMBED_DIR / 'evaluate.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
if module_path.exists():
    if module_path.read_text(encoding='utf-8') != module_source.lstrip('\n'):
        raise ValueError(f'Existing module differs from embedded source: {module_path}')
else:
    module_path.write_text(module_source.lstrip('\n'), encoding='utf-8')
print('Ready:', module_path)


In [ ]:
# Embedded source: train.py
module_source = r'''
"""Train the fixed ResNet-18 head experiments using validation macro-F1."""

from __future__ import annotations

import csv
import hashlib
import json
import os
import platform
import random
import sys
import time
from pathlib import Path

# Must be set before the first CUDA BLAS operation for repeatable head training.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
import torchvision
from torchvision.models import resnet18

from .dataset import PlantDiseaseDataset, dataset_root, is_included, load_manifest, make_transforms, result_root
from .evaluate import majority_prediction, scores, write_rows


SEED = 42
EPOCHS = 15
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHTS_ID = "ResNet18_Weights.IMAGENET1K_V1"
WEIGHTS_SHA256 = "f37072fd47e89c5e827621c5baffa7500819f7896bbacec160b1a16c560e07ec"
EXPERIMENTS = (
    ("baseline_resnet18", False, False, "frozen ResNet-18, plain cross-entropy"),
    ("augmentation_resnet18", True, False, "training augmentation only"),
    ("weighted_resnet18", False, True, "inverse-square-root class-weighted loss only"),
)
SUMMARY_COLUMNS = ["experiment", "best_epoch", "val_macro_f1", "val_accuracy",
                   "val_balanced_accuracy", "train_time_sec", "checkpoint", "change"]
HISTORY_COLUMNS = ["epoch", "train_loss", "val_loss", "train_accuracy", "val_accuracy",
                   "val_macro_f1", "epoch_time_sec"]


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def environment_info() -> dict:
    info = {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "cuda_available": torch.cuda.is_available(),
        "torch_cuda_runtime": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "gpu_vram_bytes": torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None,
        "platform": platform.platform(),
    }
    if sys.platform == "win32":
        import ctypes
        from ctypes import wintypes
        class MEMORYSTATUSEX(ctypes.Structure):
            _fields_ = [("dwLength", wintypes.DWORD), ("dwMemoryLoad", wintypes.DWORD),
                        ("ullTotalPhys", ctypes.c_ulonglong), ("ullAvailPhys", ctypes.c_ulonglong),
                        ("ullTotalPageFile", ctypes.c_ulonglong), ("ullAvailPageFile", ctypes.c_ulonglong),
                        ("ullTotalVirtual", ctypes.c_ulonglong), ("ullAvailVirtual", ctypes.c_ulonglong),
                        ("ullAvailExtendedVirtual", ctypes.c_ulonglong)]
        memory = MEMORYSTATUSEX()
        memory.dwLength = ctypes.sizeof(memory)
        if ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(memory)):
            info["ram_bytes"] = memory.ullTotalPhys
    return info


def _weights_path(output: Path) -> Path:
    path = output / "checkpoints" / "imagenet_resnet18_f37072fd.pth"
    if not path.is_file():
        raise FileNotFoundError(
            f"Official pretrained weights missing: {path}. Download from "
            "https://download.pytorch.org/models/resnet18-f37072fd.pth and place at that path."
        )
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != WEIGHTS_SHA256:
        raise ValueError(f"Official pretrained weight SHA-256 mismatch: {digest}")
    return path


def make_feature_extractor(output: Path, target_device: torch.device) -> nn.Module:
    model = resnet18(weights=None)
    state = torch.load(_weights_path(output), map_location="cpu", weights_only=True)
    model.load_state_dict(state)
    model.fc = nn.Identity()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    model.eval()
    return model.to(target_device)


def extract_features(extractor: nn.Module, dataset: PlantDiseaseDataset,
                     target_device: torch.device, batch_size: int) -> tuple[torch.Tensor, torch.Tensor]:
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0,
                        pin_memory=target_device.type == "cuda")
    features, labels = [], []
    extractor.eval()
    with torch.inference_mode():
        for images, targets in loader:
            images = images.to(target_device, non_blocking=True)
            with torch.autocast(device_type="cuda", enabled=target_device.type == "cuda"):
                values = extractor(images)
            features.append(values.float())
            labels.append(targets.to(target_device, non_blocking=True))
    return torch.cat(features), torch.cat(labels)


def _class_weights(manifest: list[dict], class_count: int, target_device: torch.device) -> tuple[torch.Tensor, list[int]]:
    counts = np.bincount([r["class_id"] for r in manifest if r["split"] == "train" and is_included(r)],
                         minlength=class_count)
    if (counts == 0).any():
        raise ValueError("A mapped class has no training examples")
    weights = 1 / np.sqrt(counts.astype(float))
    weights /= weights.mean()
    weights = np.clip(weights, 0.25, 4.0)
    return torch.tensor(weights, dtype=torch.float32, device=target_device), counts.tolist()


def _train_head_epoch(head: nn.Linear, features: torch.Tensor, labels: torch.Tensor,
                      criterion: nn.Module, optimizer: torch.optim.Optimizer,
                      batch_size: int, epoch: int) -> tuple[float, float]:
    head.train()
    generator = torch.Generator(device=features.device).manual_seed(SEED + epoch)
    order = torch.randperm(len(labels), generator=generator, device=features.device)
    total_loss, correct = 0.0, 0
    for ids in order.split(batch_size):
        optimizer.zero_grad(set_to_none=True)
        logits = head(features[ids])
        loss = criterion(logits, labels[ids])
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item()) * len(ids)
        correct += int((logits.argmax(dim=1) == labels[ids]).sum().item())
    return total_loss / len(labels), correct / len(labels)


def _validate_head(head: nn.Linear, features: torch.Tensor, labels: torch.Tensor,
                   class_count: int) -> tuple[float, dict]:
    head.eval()
    with torch.inference_mode():
        logits = head(features)
        loss = nn.functional.cross_entropy(logits, labels).item()
        prediction = logits.argmax(dim=1).cpu().numpy()
    result = scores(labels.cpu().numpy(), prediction, class_count)
    return float(loss), result


def _checkpoint(output: Path, name: str, head: nn.Linear, extractor: nn.Module,
                class_mapping: dict, best_epoch: int, val_scores: dict, config: dict) -> Path:
    model = resnet18(weights=None)
    model.fc = nn.Linear(512, len(class_mapping["classes"]))
    model.load_state_dict({**extractor.state_dict(),
                           "fc.weight": head.weight.detach().cpu(),
                           "fc.bias": head.bias.detach().cpu()})
    path = output / "checkpoints" / f"{name}_best.pth"
    torch.save({
        "model_state_dict": model.state_dict(), "architecture": "resnet18",
        "class_mapping": class_mapping, "best_epoch": best_epoch,
        "validation_macro_f1": val_scores["macro_f1"], "seed": SEED,
        "weights_identifier": WEIGHTS_ID, "pretrained_weights_sha256": WEIGHTS_SHA256,
        "preprocessing": {"resize": 256, "center_crop": 224,
                          "normalization": "ImageNet IMAGENET1K_V1 mean/std"},
        "training_config": config,
    }, path)
    return path


def train_experiments(output_dir: str | Path | None = None) -> list[dict]:
    output = result_root(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    for folder in ("checkpoints", "logs", "tables", "figures"):
        (output / folder).mkdir(exist_ok=True)
    manifest = load_manifest(output / "manifest.csv")
    class_mapping = json.loads((output / "class_mapping.json").read_text(encoding="utf-8"))
    class_count = len(class_mapping["classes"])
    fingerprint = {
        "manifest_sha256": hashlib.sha256((output / "manifest.csv").read_bytes()).hexdigest(),
        "class_mapping_sha256": hashlib.sha256((output / "class_mapping.json").read_bytes()).hexdigest(),
    }
    fingerprint_path = output / "training_data_fingerprint.json"
    if (output / "experiment_summary.csv").exists():
        if not fingerprint_path.exists() or json.loads(fingerprint_path.read_text(encoding="utf-8")) != fingerprint:
            raise ValueError("Data manifest or class mapping changed since training began; remove stale generated training artifacts before starting a new run")
    else:
        fingerprint_path.write_text(json.dumps(fingerprint, indent=2), encoding="utf-8")
    source = dataset_root()
    seed_everything()
    target_device = device()
    config = {
        "seed": SEED, "batch_size": BATCH_SIZE, "epochs": EPOCHS,
        "baseline_lr": LEARNING_RATE, "optimizer": "Adam", "weights_identifier": WEIGHTS_ID,
        "weights_sha256": WEIGHTS_SHA256, "image_size": 224,
        "validation_transform": "Resize(256), CenterCrop(224), ToTensor, ImageNet V1 Normalize",
        "augmentation": "RandomHorizontalFlip, RandomRotation(10), ColorJitter(0.1,0.1,0.1) on training images only",
        "class_weight_formula": "clip((1/sqrt(training class count))/mean(1/sqrt(training class count)), 0.25, 4.0)",
        "backbone": "frozen ResNet-18; cached deterministic features for basic and weighted experiments",
        "device": str(target_device), "num_workers": 0,
        "cublas_workspace_config": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
        "split_policy": "Original supplied train/val/test partitions; all images retained",
        "duplicate_exclusions": "None (duplicates audited and reported)",
    }
    (output / "experiment_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    (output / "environment_info.json").write_text(json.dumps(environment_info(), indent=2), encoding="utf-8")
    summary_path = output / "experiment_summary.csv"
    if summary_path.exists():
        with summary_path.open(encoding="utf-8", newline="") as handle:
            summary = list(csv.DictReader(handle))
    else:
        summary = []
    complete = {r["experiment"] for r in summary}
    if "majority_class" not in complete:
        majority = majority_prediction(manifest, "val", class_count)
        summary.append({"experiment": "majority_class", "best_epoch": 0,
                        "val_macro_f1": majority["macro_f1"], "val_accuracy": majority["accuracy"],
                        "val_balanced_accuracy": majority["balanced_accuracy"],
                        "train_time_sec": 0.0, "checkpoint": "", "change": "training-set majority class"})
        write_rows(summary_path, summary, SUMMARY_COLUMNS)
    extractor = make_feature_extractor(output, target_device)
    val_dataset = PlantDiseaseDataset(manifest, source, "val", make_transforms(False))
    val_features, val_labels = extract_features(extractor, val_dataset, target_device, BATCH_SIZE)
    train_dataset = PlantDiseaseDataset(manifest, source, "train", make_transforms(False))
    class_weights, counts = _class_weights(manifest, class_count, target_device)
    write_rows(output / "tables" / "training_class_weights.csv", [
        {"class_id": i, "class_name": class_mapping["classes"][i]["class_name"],
         "training_count": counts[i], "weight": float(class_weights[i].item())}
        for i in range(class_count)
    ])

    for name, augmented, weighted, change in EXPERIMENTS:
        if name in complete and (output / "checkpoints" / f"{name}_best.pth").exists():
            print(f"Skipping completed experiment: {name}", flush=True)
            continue
        print(f"Starting {name} on {target_device}", flush=True)
        seed_everything()
        head = nn.Linear(512, class_count).to(target_device)
        optimizer = torch.optim.Adam(head.parameters(), lr=LEARNING_RATE)
        criterion = nn.CrossEntropyLoss(weight=class_weights if weighted else None)
        deterministic_features = None
        history = []
        best_f1, best_epoch, best_metrics = -1.0, 0, None
        start = time.perf_counter()
        for epoch in range(1, EPOCHS + 1):
            epoch_start = time.perf_counter()
            if augmented:
                augmented_dataset = PlantDiseaseDataset(manifest, source, "train", make_transforms(True))
                train_features, train_labels = extract_features(extractor, augmented_dataset, target_device, BATCH_SIZE)
            else:
                if deterministic_features is None:
                    deterministic_features = extract_features(extractor, train_dataset, target_device, BATCH_SIZE)
                train_features, train_labels = deterministic_features
            train_loss, train_acc = _train_head_epoch(head, train_features, train_labels, criterion,
                                                      optimizer, BATCH_SIZE, epoch)
            val_loss, val_scores = _validate_head(head, val_features, val_labels, class_count)
            epoch_time = time.perf_counter() - epoch_start
            history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                            "train_accuracy": train_acc, "val_accuracy": val_scores["accuracy"],
                            "val_macro_f1": val_scores["macro_f1"], "epoch_time_sec": epoch_time})
            write_rows(output / "logs" / f"{name}_history.csv", history, HISTORY_COLUMNS)
            if val_scores["macro_f1"] > best_f1 + 1e-12:
                best_f1, best_epoch, best_metrics = val_scores["macro_f1"], epoch, val_scores
                _checkpoint(output, name, head, extractor, class_mapping, epoch, val_scores, config)
            print(f"{name} epoch {epoch:02d}/{EPOCHS} train_loss={train_loss:.4f} "
                  f"val_loss={val_loss:.4f} val_macro_f1={val_scores['macro_f1']:.4f} "
                  f"time={epoch_time:.1f}s", flush=True)
        elapsed = time.perf_counter() - start
        summary = [r for r in summary if r["experiment"] != name]
        summary.append({"experiment": name, "best_epoch": best_epoch,
                        "val_macro_f1": best_metrics["macro_f1"],
                        "val_accuracy": best_metrics["accuracy"],
                        "val_balanced_accuracy": best_metrics["balanced_accuracy"],
                        "train_time_sec": elapsed,
                        "checkpoint": f"checkpoints/{name}_best.pth", "change": change})
        write_rows(summary_path, summary, SUMMARY_COLUMNS)
    selected = max(summary, key=lambda row: float(row["val_macro_f1"]))["experiment"]
    summary_digest = hashlib.sha256(summary_path.read_bytes()).hexdigest()
    plan = {"selected_model": selected, "selection_metric": "validation macro-F1",
            "summary_sha256": summary_digest, "experiments": [r["experiment"] for r in summary],
            **fingerprint,
            "test_used_for_selection": False}
    (output / "frozen_plan.json").write_text(json.dumps(plan, indent=2), encoding="utf-8")
    print(f"Frozen plan: selected {selected} by validation macro-F1", flush=True)
    return summary
'''
module_path = EMBED_DIR / 'train.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
if module_path.exists():
    if module_path.read_text(encoding='utf-8') != module_source.lstrip('\n'):
        raise ValueError(f'Existing module differs from embedded source: {module_path}')
else:
    module_path.write_text(module_source.lstrip('\n'), encoding='utf-8')
print('Ready:', module_path)


In [ ]:
# Embedded source: run_experiments.py
module_source = r'''
"""Run the coursework stages. Test evaluation is an explicit final stage."""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import os
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import torch
from torchvision.models import resnet18

from .audit_dataset import audit_dataset
from .dataset import dataset_root, is_included, load_manifest, result_root
from .evaluate import majority_prediction, make_final_plots, predict_model, write_rows
from .train import WEIGHTS_ID, train_experiments, device


def _load_model(checkpoint_path: Path, mapping: dict, summary_row: dict,
                target_device: torch.device):
    payload = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    if (payload.get("architecture") != "resnet18"
            or payload.get("class_mapping") != mapping
            or payload.get("weights_identifier") != WEIGHTS_ID
            or int(payload.get("best_epoch", -1)) != int(summary_row["best_epoch"])
            or abs(float(payload.get("validation_macro_f1", -1))
                   - float(summary_row["val_macro_f1"])) > 1e-9):
        raise ValueError(f"Checkpoint does not match frozen class mapping or validation summary: {checkpoint_path}")
    model = resnet18(weights=None)
    model.fc = torch.nn.Linear(512, len(mapping["classes"]))
    model.load_state_dict(payload["model_state_dict"])
    model.eval()
    return model.to(target_device)


def final_test(output_dir: str | Path | None = None) -> dict:
    output = result_root(output_dir)
    summary_path = output / "experiment_summary.csv"
    plan_path = output / "frozen_plan.json"
    test_path = output / "final_test_metrics.json"
    if not plan_path.is_file() or not summary_path.is_file():
        raise FileNotFoundError("Train and freeze experiments before final test evaluation")
    plan = json.loads(plan_path.read_text(encoding="utf-8"))
    summary_sha = hashlib.sha256(summary_path.read_bytes()).hexdigest()
    if summary_sha != plan["summary_sha256"]:
        raise ValueError("Experiment summary changed after plan freeze; refusing test evaluation")
    for filename, key in (("manifest.csv", "manifest_sha256"),
                          ("class_mapping.json", "class_mapping_sha256")):
        actual = hashlib.sha256((output / filename).read_bytes()).hexdigest()
        if actual != plan.get(key):
            raise ValueError(f"{filename} changed after plan freeze; refusing test evaluation")
    if test_path.exists():
        existing = json.loads(test_path.read_text(encoding="utf-8"))
        if existing.get("frozen_plan_summary_sha256") != summary_sha:
            raise ValueError("Test results exist for a different frozen plan")
        print("Final test already exists for this frozen plan; reusing saved results.", flush=True)
        return existing
    with summary_path.open(encoding="utf-8", newline="") as handle:
        summary = list(csv.DictReader(handle))
    if set(plan["experiments"]) != {row["experiment"] for row in summary}:
        raise ValueError("Frozen experiment list does not match summary")
    manifest = load_manifest(output / "manifest.csv")
    mapping = json.loads((output / "class_mapping.json").read_text(encoding="utf-8"))
    class_count = len(mapping["classes"])
    target_device = device()
    test_results = {}
    selected_predictions = None
    for row in summary:
        name = row["experiment"]
        if name == "majority_class":
            metrics = majority_prediction(manifest, "test", class_count)
        else:
            checkpoint = output / row["checkpoint"]
            model = _load_model(checkpoint, mapping, row, target_device)
            metrics, predictions = predict_model(model, "test", manifest, dataset_root(),
                                                 class_count, target_device)
            if name == plan["selected_model"]:
                selected_predictions = predictions
            del model
            if target_device.type == "cuda":
                torch.cuda.empty_cache()
        test_results[name] = metrics
        print(f"Test {name}: macro-F1={metrics['macro_f1']:.4f}, "
              f"accuracy={metrics['accuracy']:.4f}, "
              f"balanced_accuracy={metrics['balanced_accuracy']:.4f}", flush=True)
    selected = plan["selected_model"]
    if selected == "majority_class":
        majority_id = test_results[selected]["majority_class_id"]
        selected_predictions = [
            {"image_id": row["image_id"], "image_path": row["image_path"],
             "true_class": int(row["class_id"]), "pred_class": majority_id, "confidence": 1.0}
            for row in manifest if row["split"] == "test" and is_included(row)
        ]
    chosen_metrics = test_results[selected]
    audit = json.loads((output / "dataset_audit.json").read_text(encoding="utf-8"))
    report = {"selected_model": selected, "selection_metric": "validation macro-F1",
              "frozen_plan_summary_sha256": summary_sha, "experiments": test_results,
              "test_set_size": len(selected_predictions),
              "release_provenance": audit.get("dataset_release", "unverified"),
              "release_version_note": audit.get("version_note", "")}
    # Test outputs are written only after all models have been evaluated under the frozen plan.
    write_rows(output / "tables" / "test_predictions.csv", selected_predictions,
               ["image_id", "image_path", "true_class", "pred_class", "confidence"])
    per_class = []
    for entry in chosen_metrics["per_class"]:
        item = dict(entry)
        item["class_name"] = mapping["classes"][item["class_id"]]["class_name"]
        per_class.append(item)
    write_rows(output / "tables" / "per_class_test.csv", per_class,
               ["class_id", "class_name", "precision", "recall", "f1", "support"])
    matrix = np.asarray(chosen_metrics["confusion_matrix"], dtype=int)
    matrix_rows = [{"true_class_id": i, **{str(j): int(matrix[i, j]) for j in range(class_count)}}
                   for i in range(class_count)]
    write_rows(output / "tables" / "confusion_matrix.csv", matrix_rows,
               ["true_class_id"] + [str(i) for i in range(class_count)])
    confusions = []
    for i in range(class_count):
        for j in range(class_count):
            if i != j and matrix[i, j] > 0:
                confusions.append({"true_class": mapping["classes"][i]["class_name"],
                                   "predicted_class": mapping["classes"][j]["class_name"],
                                   "count": int(matrix[i, j])})
    confusions.sort(key=lambda r: -r["count"])
    (output / "error_analysis.json").write_text(json.dumps({
        "top_confusions": confusions[:15],
        "error_count": int(len(selected_predictions) - np.trace(matrix)),
        "note": "Confusions are observations; symptom-level causes require visual inspection."
    }, indent=2), encoding="utf-8")
    test_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    make_final_plots(output)
    return report


def main(argv: list[str] | None = None) -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    stages = parser.add_mutually_exclusive_group(required=True)
    stages.add_argument("--audit", action="store_true", help="Build manifest and audit dataset")
    stages.add_argument("--train", action="store_true", help="Train/validate and freeze experiment plan")
    stages.add_argument("--full", action="store_true", help="Audit then train; do not evaluate test")
    stages.add_argument("--final-test", action="store_true", help="Evaluate the frozen plan on test once")
    stages.add_argument("--plots", action="store_true", help="Regenerate figures from saved results")
    args = parser.parse_args(argv)
    if args.audit or args.full:
        audit_dataset(dataset_root(), result_root())
    if args.train or args.full:
        if (result_root() / "final_test_metrics.json").exists():
            raise RuntimeError("Final test has been evaluated; do not retrain/tune against its results")
        train_experiments()
    if args.final_test:
        final_test()
    if args.plots:
        make_final_plots()


if __name__ == "__main__":
    main()
'''
module_path = EMBED_DIR / 'run_experiments.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
if module_path.exists():
    if module_path.read_text(encoding='utf-8') != module_source.lstrip('\n'):
        raise ValueError(f'Existing module differs from embedded source: {module_path}')
else:
    module_path.write_text(module_source.lstrip('\n'), encoding='utf-8')
print('Ready:', module_path)


In [ ]:
# Embedded source: inference.py
module_source = r'''
"""Load the selected checkpoint and classify one RGB image."""

from __future__ import annotations

import argparse
import json
from pathlib import Path

from PIL import Image
import torch
from torchvision.models import resnet18

from .dataset import make_transforms, result_root
from .train import device


def predict(image_path: str | Path, checkpoint_path: str | Path | None = None,
            top_k: int = 5, output_dir: str | Path | None = None) -> list[dict]:
    output = result_root(output_dir)
    if checkpoint_path is None:
        plan = json.loads((output / "frozen_plan.json").read_text(encoding="utf-8"))
        name = plan["selected_model"]
        if name == "majority_class":
            raise ValueError("The majority-class predictor has no image checkpoint")
        checkpoint_path = output / "checkpoints" / f"{name}_best.pth"
    payload = torch.load(Path(checkpoint_path), map_location="cpu", weights_only=True)
    classes = payload["class_mapping"]["classes"]
    model = resnet18(weights=None)
    model.fc = torch.nn.Linear(512, len(classes))
    model.load_state_dict(payload["model_state_dict"])
    target_device = device()
    model.to(target_device).eval()
    with Image.open(Path(image_path)) as image:
        tensor = make_transforms(False)(image.convert("RGB")).unsqueeze(0).to(target_device)
    feature_model = torch.nn.Sequential(*list(model.children())[:-1]).eval()
    with torch.inference_mode():
        with torch.autocast(device_type="cuda", enabled=target_device.type == "cuda"):
            features = feature_model(tensor)
        logits = model.fc(features.flatten(1).float())
        probabilities = torch.softmax(logits.float(), dim=1)[0]
    values, indices = probabilities.topk(min(top_k, len(classes)))
    return [
        {"class_id": int(index), "class_name": classes[int(index)]["class_name"],
         "softmax_score": float(value)}
        for value, index in zip(values.cpu().tolist(), indices.cpu().tolist())
    ]


def main(argv: list[str] | None = None) -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--image", type=Path, required=True)
    parser.add_argument("--checkpoint", type=Path, default=None)
    parser.add_argument("--top-k", type=int, default=5)
    args = parser.parse_args(argv)
    if not args.image.is_file():
        raise FileNotFoundError(args.image)
    print(json.dumps({"image": str(args.image), "predictions": predict(
        args.image, args.checkpoint, args.top_k)}, indent=2))
    print("Softmax scores are not calibrated probabilities.")


if __name__ == "__main__":
    main()
'''
module_path = EMBED_DIR / 'inference.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
if module_path.exists():
    if module_path.read_text(encoding='utf-8') != module_source.lstrip('\n'):
        raise ValueError(f'Existing module differs from embedded source: {module_path}')
else:
    module_path.write_text(module_source.lstrip('\n'), encoding='utf-8')
print('Ready:', module_path)


In [ ]:
# Embedded source: sensitivity_analysis.py
module_source = r'''
"""Post-test sensitivity check using the frozen model's saved predictions.

This does not select a model or alter the supplied PlantSeg split. It reports
how the already-frozen selected model scores after omitting test photographs
that the audit linked to a training photograph.
"""

from __future__ import annotations

import csv
import hashlib
import json
from pathlib import Path

from ..paths import PROJECT_ROOT


ROOT = PROJECT_ROOT
RESULT = ROOT / "result"


def read_csv(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))


def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def scores(rows: list[dict], class_count: int) -> dict:
    if not rows:
        raise ValueError("Sensitivity subset is empty")
    tp = [0] * class_count
    fp = [0] * class_count
    fn = [0] * class_count
    support = [0] * class_count
    for row in rows:
        true = int(row["true_class"])
        predicted = int(row["pred_class"])
        if not (0 <= true < class_count and 0 <= predicted < class_count):
            raise ValueError("Prediction has an invalid class ID")
        support[true] += 1
        if true == predicted:
            tp[true] += 1
        else:
            fn[true] += 1
            fp[predicted] += 1
    f1 = [2 * tp[i] / (2 * tp[i] + fp[i] + fn[i])
          if 2 * tp[i] + fp[i] + fn[i] else 0.0
          for i in range(class_count)]
    recall = [tp[i] / support[i] if support[i] else 0.0
              for i in range(class_count)]
    return {
        "count": len(rows),
        "classes_with_support": sum(bool(n) for n in support),
        "macro_f1": sum(f1) / class_count,
        "accuracy": sum(tp) / len(rows),
        "balanced_accuracy": sum(recall) / class_count,
    }


def main(result_dir: Path = RESULT) -> dict:
    plan = json.loads((result_dir / "frozen_plan.json").read_text(encoding="utf-8"))
    for field, filename in (("manifest_sha256", "manifest.csv"),
                            ("class_mapping_sha256", "class_mapping.json"),
                            ("summary_sha256", "experiment_summary.csv")):
        if sha256_file(result_dir / filename) != plan[field]:
            raise ValueError(f"Frozen input changed: {filename}")
    final = json.loads((result_dir / "final_test_metrics.json").read_text(encoding="utf-8"))
    if final["selected_model"] != plan["selected_model"]:
        raise ValueError("Selected model differs from the frozen plan")
    audit = json.loads((result_dir / "dataset_audit.json").read_text(encoding="utf-8"))
    mapping = json.loads((result_dir / "class_mapping.json").read_text(encoding="utf-8"))
    predictions_path = result_dir / "tables" / "test_predictions.csv"
    review_path = result_dir / "tables" / "near_duplicate_review.csv"
    if sha256_file(review_path) != audit["near_duplicate_review_sha256"]:
        raise ValueError("Visual duplicate review differs from audited review")
    predictions = read_csv(predictions_path)
    if len(predictions) != final["test_set_size"]:
        raise ValueError("Saved predictions do not cover the final test set")
    exact_test_ids = set()
    for group in audit["exact_duplicate_groups"]:
        members = group["members"]
        if any(member["split"] == "train" for member in members):
            exact_test_ids.update(member["image_id"] for member in members
                                  if member["split"] == "test")
    near_test_ids = set()
    for pair in read_csv(review_path):
        if pair["decision"] != "same_photo":
            continue
        for a, b in (("a", "b"), ("b", "a")):
            if pair[f"split_{a}"] == "train" and pair[f"split_{b}"] == "test":
                near_test_ids.add(pair[f"image_{b}"])
    overlap_ids = exact_test_ids | near_test_ids
    observed_ids = [row["image_id"] for row in predictions]
    if len(set(observed_ids)) != len(observed_ids):
        raise ValueError("Saved predictions repeat an image ID")
    if not overlap_ids.issubset(set(observed_ids)):
        raise ValueError("Audited test overlap is missing from saved predictions")
    class_count = mapping["class_count"]
    full = scores(predictions, class_count)
    recorded = final["experiments"][plan["selected_model"]]
    for metric in ("macro_f1", "accuracy", "balanced_accuracy"):
        if abs(full[metric] - recorded[metric]) > 1e-10:
            raise ValueError(f"Saved predictions disagree with final {metric}")
    clean = scores([row for row in predictions if row["image_id"] not in overlap_ids],
                   class_count)
    output = {
        "purpose": "Post-hoc sensitivity analysis only; no model selection or retraining",
        "selected_model": plan["selected_model"],
        "definition": "Omit test images with an exact or visually confirmed same-photo match in training",
        "official_split_unchanged": True,
        "exact_test_train_overlap": len(exact_test_ids),
        "near_test_train_overlap": len(near_test_ids),
        "overlap_union": len(overlap_ids),
        "all_test": full,
        "without_detected_train_overlap": clean,
        "remaining_limitation": "Undetected overlaps may remain; validation also has audited overlaps",
        "predictions_sha256": sha256_file(predictions_path),
        "review_sha256": sha256_file(review_path),
    }
    destination = result_dir / "overlap_sensitivity.json"
    destination.write_text(json.dumps(output, indent=2) + "\n", encoding="utf-8")
    print(f"Saved {destination}: {full['count']} full, {clean['count']} without detected train overlap")
    return output


if __name__ == "__main__":
    main()
'''
module_path = EMBED_DIR / 'analysis/sensitivity_analysis.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
if module_path.exists():
    if module_path.read_text(encoding='utf-8') != module_source.lstrip('\n'):
        raise ValueError(f'Existing module differs from embedded source: {module_path}')
else:
    module_path.write_text(module_source.lstrip('\n'), encoding='utf-8')
print('Ready:', module_path)


In [ ]:
# Embedded source: explore_finetune.py
module_source = r'''
"""Validation-only, post-final pilot: fine-tune ResNet-18 layer4 and head.

The original frozen plan and test outputs are deliberately untouched. This
script reads only training and validation images and writes under a separate
exploratory directory. Its results cannot replace the original final test.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import time
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision.models import resnet18

from ..dataset import PlantDiseaseDataset, dataset_root, load_manifest, make_transforms, result_root
from ..evaluate import scores
from ..train import BATCH_SIZE, SEED, WEIGHTS_ID, device, seed_everything


def digest(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def main(epochs: int = 10, patience: int = 4, output_dir: Path | None = None,
         continue_from_best_pilot: bool = False) -> dict:
    output = result_root(output_dir)
    plan = json.loads((output / "frozen_plan.json").read_text(encoding="utf-8"))
    for key, name in (("manifest_sha256", "manifest.csv"),
                      ("class_mapping_sha256", "class_mapping.json"),
                      ("summary_sha256", "experiment_summary.csv")):
        if plan[key] != digest(output / name):
            raise ValueError(f"Frozen input changed: {name}")
    mapping = json.loads((output / "class_mapping.json").read_text(encoding="utf-8"))
    manifest = load_manifest(output / "manifest.csv")
    baseline_path = output / "checkpoints" / "baseline_resnet18_best.pth"
    original_baseline = torch.load(baseline_path, map_location="cpu", weights_only=True)
    if original_baseline["class_mapping"] != mapping or original_baseline["weights_identifier"] != WEIGHTS_ID:
        raise ValueError("Baseline checkpoint does not match frozen data or pretrained weights")
    with (output / "experiment_summary.csv").open(encoding="utf-8", newline="") as handle:
        row = next(r for r in csv.DictReader(handle) if r["experiment"] == "baseline_resnet18")
    if (int(row["best_epoch"]) != original_baseline["best_epoch"] or
            abs(float(row["val_macro_f1"]) - original_baseline["validation_macro_f1"]) > 1e-10):
        raise ValueError("Baseline checkpoint differs from frozen validation summary")
    initial_path = baseline_path
    initial_scores = {"macro_f1": float(row["val_macro_f1"]),
                      "accuracy": float(row["val_accuracy"]),
                      "balanced_accuracy": float(row["val_balanced_accuracy"])}
    if continue_from_best_pilot:
        pilot_dir = output / "exploratory_finetune"
        pilot = json.loads((pilot_dir / "summary.json").read_text(encoding="utf-8"))
        if pilot["best_epoch"] < 1 or not pilot["checkpoint"]:
            raise ValueError("Pilot has no improved checkpoint to continue")
        initial_path = output / pilot["checkpoint"]
        initial_scores = pilot["best_validation"]
    baseline = torch.load(initial_path, map_location="cpu", weights_only=True)
    if baseline["class_mapping"] != mapping or baseline["weights_identifier"] != WEIGHTS_ID:
        raise ValueError("Continuation checkpoint does not match frozen data")

    seed_everything(SEED)
    target = device()
    source = dataset_root()
    train_set = PlantDiseaseDataset(manifest, source, "train", make_transforms(False))
    val_set = PlantDiseaseDataset(manifest, source, "val", make_transforms(False))
    if len(train_set) != 3007 or len(val_set) != 475:
        raise ValueError("Unexpected original training/validation split size")
    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                              generator=generator, num_workers=0, pin_memory=target.type == "cuda")
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=0, pin_memory=target.type == "cuda")
    model = resnet18(weights=None)
    model.fc = nn.Linear(512, mapping["class_count"])
    model.load_state_dict(baseline["model_state_dict"])
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    for parameter in model.layer4.parameters():
        parameter.requires_grad_(True)
    for parameter in model.fc.parameters():
        parameter.requires_grad_(True)
    model.to(target)
    optimizer = torch.optim.Adam([
        {"params": model.layer4.parameters(), "lr": 1e-5},
        {"params": model.fc.parameters(), "lr": 1e-4},
    ])
    criterion = nn.CrossEntropyLoss()
    amp = target.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    folder = output / ("exploratory_finetune_continued" if continue_from_best_pilot
                       else "exploratory_finetune")
    folder.mkdir(parents=True, exist_ok=True)
    history_path = folder / "validation_history.csv"
    checkpoint_path = folder / "best_validation.pth"
    config = {
        "status": "post-final validation-only exploration; do not use for frozen final test",
        "stage": "continued from best pilot with freshly initialized Adam optimizer"
                 if continue_from_best_pilot else "initial fine-tuning pilot",
        "initial_checkpoint": str(initial_path.relative_to(output)).replace("\\", "/"),
        "initial_checkpoint_sha256": digest(initial_path),
        "initial_validation": initial_scores,
        "manifest_sha256": plan["manifest_sha256"],
        "class_mapping_sha256": plan["class_mapping_sha256"],
        "seed": SEED, "max_epochs": epochs, "patience": patience,
        "train_size": len(train_set), "validation_size": len(val_set),
        "batch_size": BATCH_SIZE, "optimizer": "Adam",
        "layer4_lr": 1e-5, "head_lr": 1e-4, "loss": "plain cross entropy",
        "augmentation": "none; same deterministic train/validation transform as original baseline",
        "trainable": "layer4 and fc", "device": str(target), "amp": amp,
        "checkpoint_rule": "highest validation macro-F1; first epoch wins exact ties",
    }
    (folder / "config.json").write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
    best_f1 = float(initial_scores["macro_f1"])
    best_epoch = 0
    best_scores = initial_scores
    history = []
    no_gain = 0
    start = time.perf_counter()
    for epoch in range(1, epochs + 1):
        epoch_start = time.perf_counter()
        model.train()
        # Frozen early blocks keep their ImageNet batch-normalization statistics.
        for module in (model.bn1, *[m for block in (model.layer1, model.layer2, model.layer3)
                                      for m in block.modules() if isinstance(m, nn.BatchNorm2d)]):
            if isinstance(module, nn.BatchNorm2d):
                module.eval()
        train_loss = 0.0
        train_correct = 0
        for images, labels in train_loader:
            images = images.to(target, non_blocking=True)
            labels = labels.to(target, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", enabled=amp):
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += float(loss.item()) * len(labels)
            train_correct += int((logits.argmax(1) == labels).sum().item())
        model.eval()
        val_true, val_pred = [], []
        val_loss = 0.0
        with torch.inference_mode():
            for images, labels in val_loader:
                images = images.to(target, non_blocking=True)
                labels = labels.to(target, non_blocking=True)
                with torch.autocast(device_type="cuda", enabled=amp):
                    logits = model(images)
                    loss = criterion(logits, labels)
                val_loss += float(loss.item()) * len(labels)
                val_true.extend(labels.cpu().tolist())
                val_pred.extend(logits.argmax(1).cpu().tolist())
        measured = scores(val_true, val_pred, mapping["class_count"])
        record = {
            "epoch": epoch, "train_loss": train_loss / len(train_set),
            "train_accuracy": train_correct / len(train_set),
            "val_loss": val_loss / len(val_set),
            "val_macro_f1": measured["macro_f1"],
            "val_accuracy": measured["accuracy"],
            "val_balanced_accuracy": measured["balanced_accuracy"],
            "epoch_time_sec": time.perf_counter() - epoch_start,
        }
        history.append(record)
        with history_path.open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(record))
            writer.writeheader()
            writer.writerows(history)
        if measured["macro_f1"] > best_f1 + 1e-12:
            best_f1 = measured["macro_f1"]
            best_epoch = epoch
            best_scores = {key: measured[key] for key in
                           ("macro_f1", "accuracy", "balanced_accuracy")}
            no_gain = 0
            torch.save({"model_state_dict": model.state_dict(),
                        "class_mapping": mapping, "architecture": "resnet18",
                        "best_epoch": epoch, "validation_macro_f1": best_f1,
                        "pretrained_weights_sha256": baseline["pretrained_weights_sha256"],
                        "weights_identifier": WEIGHTS_ID,
                        "preprocessing": baseline["preprocessing"],
                        "training_config": config}, checkpoint_path)
        else:
            no_gain += 1
        print(f"epoch {epoch:02d}/{epochs} train_loss={record['train_loss']:.4f} "
              f"val_macro_f1={measured['macro_f1']:.4f} best={best_f1:.4f} "
              f"time={record['epoch_time_sec']:.1f}s", flush=True)
        if no_gain >= patience:
            print(f"Stopped after {patience} epochs without validation macro-F1 gain", flush=True)
            break
    summary = {"status": config["status"], "stage": config["stage"],
               "baseline_validation_macro_f1": float(row["val_macro_f1"]),
               "initial_validation_macro_f1": float(initial_scores["macro_f1"]),
               "best_epoch": best_epoch, "best_validation": best_scores,
               "epochs_run": len(history), "training_time_sec": time.perf_counter() - start,
               "checkpoint": str(checkpoint_path.relative_to(output)).replace("\\", "/")
               if best_epoch else None,
               "best_available_checkpoint": str((checkpoint_path if best_epoch else initial_path)
                                                 .relative_to(output)).replace("\\", "/"),
               "test_evaluated": False}
    (folder / "summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(summary, indent=2), flush=True)
    return summary


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--epochs", type=int, default=10)
    parser.add_argument("--patience", type=int, default=4)
    parser.add_argument("--continue-from-best-pilot", action="store_true")
    args = parser.parse_args()
    main(args.epochs, args.patience, continue_from_best_pilot=args.continue_from_best_pilot)
'''
module_path = EMBED_DIR / 'analysis/explore_finetune.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
if module_path.exists():
    if module_path.read_text(encoding='utf-8') != module_source.lstrip('\n'):
        raise ValueError(f'Existing module differs from embedded source: {module_path}')
else:
    module_path.write_text(module_source.lstrip('\n'), encoding='utf-8')
print('Ready:', module_path)


In [ ]:
# Embedded source: validation_overlap_analysis.py
module_source = r'''
"""Check frozen validation predictions with detected train-photo overlaps omitted.

The original selection and full-split scores remain unchanged. This script
evaluates saved frozen checkpoints on validation only and records a sensitivity
analysis for interpretation; it never reads the test split.
"""

from __future__ import annotations

import csv
import hashlib
import json
from pathlib import Path

import torch
from torchvision.models import resnet18

from ..dataset import dataset_root, load_manifest, result_root
from ..evaluate import predict_model, scores
from ..train import WEIGHTS_ID, device


def read_rows(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def sha(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def main(output_dir: Path | None = None) -> dict:
    output = result_root(output_dir)
    plan = json.loads((output / "frozen_plan.json").read_text(encoding="utf-8"))
    for field, filename in (("manifest_sha256", "manifest.csv"),
                            ("class_mapping_sha256", "class_mapping.json"),
                            ("summary_sha256", "experiment_summary.csv")):
        if sha(output / filename) != plan[field]:
            raise ValueError(f"Frozen input changed: {filename}")
    audit = json.loads((output / "dataset_audit.json").read_text(encoding="utf-8"))
    review_path = output / "tables" / "near_duplicate_review.csv"
    if sha(review_path) != audit["near_duplicate_review_sha256"]:
        raise ValueError("Audited visual review changed")
    mapping = json.loads((output / "class_mapping.json").read_text(encoding="utf-8"))
    manifest = load_manifest(output / "manifest.csv")
    val_rows = [row for row in manifest if row["split"] == "val"]
    exact = set()
    for group in audit["exact_duplicate_groups"]:
        members = group["members"]
        if any(member["split"] == "train" for member in members):
            exact.update(member["image_id"] for member in members
                         if member["split"] == "val")
    near = set()
    for pair in read_rows(review_path):
        if pair["decision"] != "same_photo":
            continue
        if pair["split_a"] == "train" and pair["split_b"] == "val":
            near.add(pair["image_b"])
        if pair["split_b"] == "train" and pair["split_a"] == "val":
            near.add(pair["image_a"])
    overlap = exact | near
    if not overlap.issubset({row["image_id"] for row in val_rows}):
        raise ValueError("Audit references unknown validation images")
    summary_rows = read_rows(output / "experiment_summary.csv")
    pilot_path = output / "exploratory_finetune" / "summary.json"
    continued_path = output / "exploratory_finetune_continued" / "summary.json"
    if pilot_path.is_file():
        pilot = json.loads(pilot_path.read_text(encoding="utf-8"))
        latest = (json.loads(continued_path.read_text(encoding="utf-8"))
                  if continued_path.is_file() else pilot)
        if latest.get("best_available_checkpoint"):
            checkpoint = latest["best_available_checkpoint"]
            expected = latest["best_validation"]["macro_f1"]
        else:
            checkpoint = pilot["checkpoint"]
            expected = pilot["best_validation"]["macro_f1"]
        summary_rows.append({"experiment": "finetune_layer4_postfinal",
                             "checkpoint": checkpoint,
                             "best_epoch": str(pilot["best_epoch"]),
                             "val_macro_f1": expected})
    target = device()
    prediction_rows = {row["image_id"]: {
        "image_id": row["image_id"], "true_class": int(row["class_id"]),
        "train_photo_overlap": int(row["image_id"] in overlap),
    } for row in val_rows}
    results = {}
    for entry in summary_rows:
        name = entry["experiment"]
        if name == "majority_class":
            continue
        payload = torch.load(output / entry["checkpoint"], map_location="cpu", weights_only=True)
        if (payload["class_mapping"] != mapping or payload["weights_identifier"] != WEIGHTS_ID
                or abs(float(entry["val_macro_f1"]) - payload["validation_macro_f1"]) > 1e-10):
            raise ValueError(f"Frozen checkpoint does not match: {name}")
        model = resnet18(weights=None)
        model.fc = torch.nn.Linear(512, mapping["class_count"])
        model.load_state_dict(payload["model_state_dict"])
        model.to(target).eval()
        full, predictions = predict_model(model, "val", manifest, dataset_root(),
                                          mapping["class_count"], target)
        if abs(full["macro_f1"] - float(entry["val_macro_f1"])) > 1e-6:
            raise ValueError(f"Recomputed validation result differs from frozen summary: {name}")
        for row in predictions:
            prediction_rows[row["image_id"]][name] = row["pred_class"]
        clean_predictions = [row for row in predictions if row["image_id"] not in overlap]
        clean = scores([row["true_class"] for row in clean_predictions],
                       [row["pred_class"] for row in clean_predictions], mapping["class_count"])
        results[name] = {
            "all_validation": {key: full[key] for key in ("macro_f1", "accuracy", "balanced_accuracy")},
            "without_detected_train_overlap": {key: clean[key] for key in
                                                 ("macro_f1", "accuracy", "balanced_accuracy")},
        }
        print(f"{name}: full {full['macro_f1']:.4f}, without overlap {clean['macro_f1']:.4f}", flush=True)
        del model
        if target.type == "cuda":
            torch.cuda.empty_cache()
    table_path = output / "tables" / "validation_overlap_predictions.csv"
    columns = ["image_id", "true_class", "train_photo_overlap"] + list(results)
    with table_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(prediction_rows[row["image_id"]] for row in val_rows)
    report = {
        "status": "interpretive sensitivity analysis; original selection unchanged",
        "exact_validation_train_overlap": len(exact),
        "near_validation_train_overlap": len(near),
        "overlap_union": len(overlap),
        "all_validation_count": len(val_rows),
        "without_overlap_count": len(val_rows) - len(overlap),
        "experiments": results,
        "predictions_sha256": sha(table_path),
        "test_evaluated": False,
    }
    (output / "validation_overlap_analysis.json").write_text(
        json.dumps(report, indent=2) + "\n", encoding="utf-8")
    return report


if __name__ == "__main__":
    main()
'''
module_path = EMBED_DIR / 'analysis/validation_overlap_analysis.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
if module_path.exists():
    if module_path.read_text(encoding='utf-8') != module_source.lstrip('\n'):
        raise ValueError(f'Existing module differs from embedded source: {module_path}')
else:
    module_path.write_text(module_source.lstrip('\n'), encoding='utf-8')
print('Ready:', module_path)


In [ ]:
# Embedded source: evaluate_extension.py
module_source = r'''
"""Freeze and evaluate the post-final fine-tuning extension once.

This is explicitly separate from the original frozen test plan. Prior test
results already existed when this extension was designed, so this evaluation
must be labelled supplementary and cannot be presented as a blind final test.
"""

from __future__ import annotations

import argparse
import hashlib
import json
from pathlib import Path

import torch
from torchvision.models import resnet18

from ..dataset import dataset_root, load_manifest, result_root
from ..evaluate import predict_model, scores, write_rows
from .sensitivity_analysis import read_csv
from ..train import WEIGHTS_ID, device


def sha(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def extension_folder(output: Path) -> Path:
    return output / "exploratory_finetune"


def freeze(output: Path) -> dict:
    folder = extension_folder(output)
    plan_path = folder / "frozen_extension_plan.json"
    if plan_path.exists():
        raise FileExistsError("Extension plan already frozen")
    original = json.loads((output / "frozen_plan.json").read_text(encoding="utf-8"))
    summary_path = folder / "summary.json"
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    continued_path = output / "exploratory_finetune_continued" / "summary.json"
    continued = json.loads(continued_path.read_text(encoding="utf-8"))
    if continued["best_validation"]["macro_f1"] != summary["best_validation"]["macro_f1"]:
        raise ValueError("Continuation has a different best validation score")
    if continued["best_available_checkpoint"] != summary["checkpoint"]:
        raise ValueError("Continuation best differs from pilot best checkpoint")
    checkpoint = output / summary["checkpoint"]
    payload = torch.load(checkpoint, map_location="cpu", weights_only=True)
    mapping = json.loads((output / "class_mapping.json").read_text(encoding="utf-8"))
    if (payload["class_mapping"] != mapping or payload["weights_identifier"] != WEIGHTS_ID
            or payload["best_epoch"] != summary["best_epoch"]
            or abs(payload["validation_macro_f1"] - summary["best_validation"]["macro_f1"]) > 1e-10):
        raise ValueError("Checkpoint does not match validation-selected extension")
    for field, filename in (("manifest_sha256", "manifest.csv"),
                            ("class_mapping_sha256", "class_mapping.json"),
                            ("summary_sha256", "experiment_summary.csv")):
        if original[field] != sha(output / filename):
            raise ValueError(f"Original frozen input changed: {filename}")
    validation = json.loads((output / "validation_overlap_analysis.json").read_text(encoding="utf-8"))
    if abs(validation["experiments"]["finetune_layer4_postfinal"]["all_validation"]["macro_f1"]
           - summary["best_validation"]["macro_f1"]) > 1e-10:
        raise ValueError("Independent validation prediction check differs")
    plan = {
        "label": "supplementary post-final extension; prior test results were already visible",
        "selection_rule": "highest validation macro-F1 across fine-tuning pilot and continuation",
        "selected_checkpoint": summary["checkpoint"],
        "checkpoint_sha256": sha(checkpoint),
        "validation_macro_f1": summary["best_validation"]["macro_f1"],
        "pilot_best_epoch": summary["best_epoch"],
        "continued_epochs_without_improvement": continued["epochs_run"],
        "original_frozen_plan_sha256": sha(output / "frozen_plan.json"),
        "manifest_sha256": original["manifest_sha256"],
        "class_mapping_sha256": original["class_mapping_sha256"],
        "validation_analysis_sha256": sha(output / "validation_overlap_analysis.json"),
        "prior_test_results_available": (output / "final_test_metrics.json").exists(),
        "test_used_to_select_this_checkpoint": False,
    }
    plan_path.write_text(json.dumps(plan, indent=2) + "\n", encoding="utf-8")
    print(f"Frozen supplementary extension: {summary['checkpoint']} at validation macro-F1 "
          f"{plan['validation_macro_f1']:.4f}")
    return plan


def evaluate(output: Path) -> dict:
    folder = extension_folder(output)
    plan_path = folder / "frozen_extension_plan.json"
    test_path = folder / "supplementary_test_metrics.json"
    if test_path.exists():
        raise FileExistsError("Supplementary test has already been evaluated")
    plan = json.loads(plan_path.read_text(encoding="utf-8"))
    for field, path in (("checkpoint_sha256", output / plan["selected_checkpoint"]),
                        ("manifest_sha256", output / "manifest.csv"),
                        ("class_mapping_sha256", output / "class_mapping.json"),
                        ("validation_analysis_sha256", output / "validation_overlap_analysis.json"),
                        ("original_frozen_plan_sha256", output / "frozen_plan.json")):
        if sha(path) != plan[field]:
            raise ValueError(f"Frozen extension input changed: {path.name}")
    mapping = json.loads((output / "class_mapping.json").read_text(encoding="utf-8"))
    payload = torch.load(output / plan["selected_checkpoint"], map_location="cpu", weights_only=True)
    model = resnet18(weights=None)
    model.fc = torch.nn.Linear(512, mapping["class_count"])
    model.load_state_dict(payload["model_state_dict"])
    target = device()
    model.to(target).eval()
    full, predictions = predict_model(model, "test", load_manifest(output / "manifest.csv"),
                                      dataset_root(), mapping["class_count"], target)
    if len(predictions) != 867:
        raise ValueError("Expected all 867 official test images")
    prediction_path = folder / "supplementary_test_predictions.csv"
    write_rows(prediction_path, predictions,
               ["image_id", "image_path", "true_class", "pred_class", "confidence"])
    audit = json.loads((output / "dataset_audit.json").read_text(encoding="utf-8"))
    exact = set()
    for group in audit["exact_duplicate_groups"]:
        members = group["members"]
        if any(member["split"] == "train" for member in members):
            exact.update(member["image_id"] for member in members if member["split"] == "test")
    near = set()
    review_path = output / "tables" / "near_duplicate_review.csv"
    if sha(review_path) != audit["near_duplicate_review_sha256"]:
        raise ValueError("Audited duplicate review changed")
    for pair in read_csv(review_path):
        if pair["decision"] != "same_photo":
            continue
        if pair["split_a"] == "train" and pair["split_b"] == "test":
            near.add(pair["image_b"])
        if pair["split_b"] == "train" and pair["split_a"] == "test":
            near.add(pair["image_a"])
    overlap = exact | near
    clean_rows = [row for row in predictions if row["image_id"] not in overlap]
    clean = scores([row["true_class"] for row in clean_rows],
                   [row["pred_class"] for row in clean_rows], mapping["class_count"])
    output_metrics = {
        "label": plan["label"],
        "frozen_extension_plan_sha256": sha(plan_path),
        "prior_test_results_available": plan["prior_test_results_available"],
        "test_used_to_select_this_checkpoint": False,
        "all_official_test": {key: full[key] for key in
                              ("macro_f1", "accuracy", "balanced_accuracy", "per_class", "confusion_matrix")},
        "without_detected_train_photo_overlap": {key: clean[key] for key in
                                                  ("macro_f1", "accuracy", "balanced_accuracy")},
        "overlap_union": len(overlap),
        "remaining_count": len(clean_rows),
        "predictions_sha256": sha(prediction_path),
    }
    test_path.write_text(json.dumps(output_metrics, indent=2) + "\n", encoding="utf-8")
    print(f"Supplementary test macro-F1 {full['macro_f1']:.4f}; "
          f"{len(clean_rows)} without detected train overlap: {clean['macro_f1']:.4f}")
    return output_metrics


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    group = parser.add_mutually_exclusive_group(required=True)
    group.add_argument("--freeze", action="store_true")
    group.add_argument("--test", action="store_true")
    args = parser.parse_args()
    output = result_root()
    freeze(output) if args.freeze else evaluate(output)
'''
module_path = EMBED_DIR / 'analysis/evaluate_extension.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
if module_path.exists():
    if module_path.read_text(encoding='utf-8') != module_source.lstrip('\n'):
        raise ValueError(f'Existing module differs from embedded source: {module_path}')
else:
    module_path.write_text(module_source.lstrip('\n'), encoding='utf-8')
print('Ready:', module_path)


In [ ]:
# Embedded source: compare_extension.py
module_source = r'''
"""Compare the frozen original model with the supplementary fine-tuned model.

Uses only saved, paired predictions. The bootstrap interval is descriptive:
the extension was designed after original test results were already visible,
and original split overlaps can reduce effective sample independence.
"""

from __future__ import annotations

import csv
import hashlib
import json
from pathlib import Path

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from ..dataset import result_root


def sha(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def read_csv(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def macro_f1(true: np.ndarray, pred: np.ndarray, count: int) -> float:
    matrix = np.bincount(true * count + pred, minlength=count**2).reshape(count, count)
    correct = matrix.diagonal()
    denominator = matrix.sum(axis=0) + matrix.sum(axis=1)
    return float(np.divide(2 * correct, denominator, out=np.zeros(count, dtype=float),
                           where=denominator > 0).mean())


def plot_extension(output: Path, summary: dict) -> None:
    folder = output / "exploratory_finetune"
    pilot_summary = json.loads((folder / "summary.json").read_text(encoding="utf-8"))
    with (output / "experiment_summary.csv").open("r", encoding="utf-8", newline="") as handle:
        weighted_validation = float(next(row for row in csv.DictReader(handle)
                                         if row["experiment"] == "weighted_resnet18")["val_macro_f1"])
    pilot = read_csv(folder / "validation_history.csv")
    continued = read_csv(output / "exploratory_finetune_continued" / "validation_history.csv")
    history = pilot + continued
    epochs = np.arange(1, len(history) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
    axes[0].plot(epochs, [float(r["train_loss"]) for r in history], marker="o", ms=3,
                 label="Training loss")
    axes[0].plot(epochs, [float(r["val_loss"]) for r in history], marker="o", ms=3,
                 label="Validation loss")
    axes[0].set(ylabel="Cross-entropy", xlabel="Epoch", title="Fine-tuning loss")
    axes[0].legend(frameon=False, fontsize=8)
    axes[1].plot(epochs, [float(r["val_macro_f1"]) for r in history], marker="o", ms=4,
                 color="#157a75", label="Fine-tuning")
    axes[1].axhline(weighted_validation, ls="--", color="#536475", lw=1,
                    label="Original class weighting")
    axes[1].scatter([pilot_summary["best_epoch"]],
                    [pilot_summary["best_validation"]["macro_f1"]], color="#bd4c35", zorder=5,
                    label="Selected checkpoint")
    axes[1].set(ylabel="Macro-F1", xlabel="Epoch", title="Validation selection")
    axes[1].legend(frameon=False, fontsize=8)
    for ax in axes:
        ax.axvline(len(pilot) + .5, color="#b5bec5", ls=":", lw=1)
        ax.grid(alpha=.2)
    fig.suptitle("The continuation restarted Adam from the pilot's best checkpoint", fontsize=10)
    fig.tight_layout()
    fig.savefig(output / "figures" / "finetune_curves.png", dpi=170)
    plt.close(fig)

    names = ["Macro-F1", "Accuracy", "Balanced acc."]
    keys = ["macro_f1", "accuracy", "balanced_accuracy"]
    original = [summary["metrics"][k]["original"] for k in keys]
    fine = [summary["metrics"][k]["finetune"] for k in keys]
    x = np.arange(3)
    fig, ax = plt.subplots(figsize=(7.5, 3.6))
    ax.bar(x - .18, original, width=.36, label="Original class weighting", color="#536475")
    ax.bar(x + .18, fine, width=.36, label="Supplementary fine-tuning", color="#157a75")
    ax.set(xticks=x, xticklabels=names, ylabel="Score", ylim=(0, .80),
           title="Official test split: all 867 images")
    ax.legend(frameon=False, fontsize=8)
    ax.grid(axis="y", alpha=.18)
    for i, value in enumerate(original):
        ax.text(i - .18, value + .012, f"{value:.3f}", ha="center", fontsize=8)
    for i, value in enumerate(fine):
        ax.text(i + .18, value + .012, f"{value:.3f}", ha="center", fontsize=8)
    fig.tight_layout()
    fig.savefig(output / "figures" / "extension_comparison.png", dpi=170)
    plt.close(fig)


def main(output_dir: Path | None = None, bootstrap_replicates: int = 3000) -> dict:
    output = result_root(output_dir)
    folder = output / "exploratory_finetune"
    original = json.loads((output / "final_test_metrics.json").read_text(encoding="utf-8"))
    extension = json.loads((folder / "supplementary_test_metrics.json").read_text(encoding="utf-8"))
    plan_path = folder / "frozen_extension_plan.json"
    if extension["frozen_extension_plan_sha256"] != sha(plan_path):
        raise ValueError("Supplementary test is detached from its frozen plan")
    original_path = output / "tables" / "test_predictions.csv"
    extension_path = folder / "supplementary_test_predictions.csv"
    if extension["predictions_sha256"] != sha(extension_path):
        raise ValueError("Supplementary predictions changed")
    first = {row["image_id"]: row for row in read_csv(original_path)}
    second = read_csv(extension_path)
    if len(first) != 867 or len(second) != 867 or set(first) != {r["image_id"] for r in second}:
        raise ValueError("The paired test predictions do not cover the same 867 images")
    mapping = json.loads((output / "class_mapping.json").read_text(encoding="utf-8"))
    count = mapping["class_count"]
    true = np.array([int(row["true_class"]) for row in second])
    earlier = np.array([int(first[row["image_id"]]["pred_class"]) for row in second])
    later = np.array([int(row["pred_class"]) for row in second])
    if any(int(first[row["image_id"]]["true_class"]) != int(row["true_class"])
           for row in second):
        raise ValueError("The two test prediction tables use different labels")
    original_metrics = original["experiments"][original["selected_model"]]
    extension_metrics = extension["all_official_test"]
    if (abs(macro_f1(true, earlier, count) - original_metrics["macro_f1"]) > 1e-10
            or abs(macro_f1(true, later, count) - extension_metrics["macro_f1"]) > 1e-10):
        raise ValueError("Paired prediction scores differ from saved test metrics")
    by_class = [np.where(true == class_id)[0] for class_id in range(count)]
    generator = np.random.default_rng(2026)
    differences = np.empty(bootstrap_replicates)
    for index in range(bootstrap_replicates):
        sample = np.concatenate([generator.choice(ids, size=len(ids), replace=True)
                                 for ids in by_class])
        differences[index] = (macro_f1(true[sample], later[sample], count)
                              - macro_f1(true[sample], earlier[sample], count))
    original_per_class = original_metrics["per_class"]
    new_per_class = extension_metrics["per_class"]
    comparison_rows = []
    for class_id in range(count):
        old = original_per_class[class_id]
        new = new_per_class[class_id]
        if old["support"] != new["support"]:
            raise ValueError("Per-class support differs between models")
        comparison_rows.append({
            "class_id": class_id,
            "class_name": mapping["classes"][class_id]["class_name"],
            "test_support": old["support"],
            "original_precision": old["precision"], "finetune_precision": new["precision"],
            "original_recall": old["recall"], "finetune_recall": new["recall"],
            "original_f1": old["f1"], "finetune_f1": new["f1"],
            "f1_change": new["f1"] - old["f1"],
        })
    with (folder / "per_class_test_comparison.csv").open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(comparison_rows[0]))
        writer.writeheader()
        writer.writerows(comparison_rows)
    matrix = extension_metrics["confusion_matrix"]
    with (folder / "confusion_matrix.csv").open("w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["true_class_id"] + list(range(count)))
        for class_id, row in enumerate(matrix):
            writer.writerow([class_id] + row)
    summary = {
        "label": "supplementary post-final comparison; extension was not blind to earlier test results",
        "test_split_unchanged": True,
        "sample_count": len(true),
        "original_model": original["selected_model"],
        "finetune_model": "layer4 plus head, epoch 10 checkpoint",
        "metrics": {
            name: {
                "original": original_metrics[name],
                "finetune": extension_metrics[name],
                "change": extension_metrics[name] - original_metrics[name],
            } for name in ("macro_f1", "accuracy", "balanced_accuracy")
        },
        "finetune_only_correct": int(np.sum((later == true) & (earlier != true))),
        "original_only_correct": int(np.sum((earlier == true) & (later != true))),
        "paired_stratified_bootstrap": {
            "seed": 2026,
            "replicates": bootstrap_replicates,
            "macro_f1_change_95_percent_interval": np.quantile(differences, [.025, .975]).tolist(),
            "interpretation": "Descriptive only: prior test visibility and duplicate overlap limit inference",
        },
        "largest_class_gains": [row["class_name"] for row in
                                sorted(comparison_rows, key=lambda r: r["f1_change"], reverse=True)[:5]],
        "largest_class_losses": [row["class_name"] for row in
                                 sorted(comparison_rows, key=lambda r: r["f1_change"])[:5]],
        "original_predictions_sha256": sha(original_path),
        "finetune_predictions_sha256": sha(extension_path),
    }
    (folder / "comparison.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    plot_extension(output, summary)
    print(f"Paired test macro-F1 change {summary['metrics']['macro_f1']['change']:.4f}; "
          f"95% descriptive bootstrap interval "
          f"{summary['paired_stratified_bootstrap']['macro_f1_change_95_percent_interval']}")
    return summary


if __name__ == "__main__":
    main()
'''
module_path = EMBED_DIR / 'analysis/compare_extension.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
if module_path.exists():
    if module_path.read_text(encoding='utf-8') != module_source.lstrip('\n'):
        raise ValueError(f'Existing module differs from embedded source: {module_path}')
else:
    module_path.write_text(module_source.lstrip('\n'), encoding='utf-8')
print('Ready:', module_path)


In [ ]:
from pathlib import Path
import csv
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
from collections import Counter

MODE = "auto"  # Choices: "full", "demo", or "recorded".
assert MODE in {"auto", "full", "demo", "recorded"}

candidates = [Path.cwd(), Path.cwd() / "code", Path.cwd().parent / "code",
              Path("/content/code"), Path("/kaggle/working/code")]
CODE_DIR = next((p.resolve() for p in candidates
                 if (p / "plantseg_cw2" / "dataset.py").is_file()), None)
if CODE_DIR is None:
    raise FileNotFoundError("Put the notebook beside the code folder or in it.")
PROJECT_ROOT = CODE_DIR.parent
archived_result = PROJECT_ROOT / "result"
if MODE == "auto":
    plan_path = archived_result / "frozen_plan.json"
    if (archived_result / "final_test_metrics.json").is_file() and plan_path.is_file():
        selected = json.loads(plan_path.read_text(encoding="utf-8"))["selected_model"]
        checkpoint = archived_result / "checkpoints" / f"{selected}_best.pth"
        MODE = "demo" if checkpoint.is_file() else "recorded"
    else:
        MODE = "full"
RESULT_DIR = archived_result
if MODE == "full" and (archived_result / "final_test_metrics.json").is_file():
    RESULT_DIR = PROJECT_ROOT / "result_local"
    review_source = archived_result / "tables" / "near_duplicate_review.csv"
    review_target = RESULT_DIR / "tables" / "near_duplicate_review.csv"
    if review_source.is_file() and not review_target.is_file():
        review_target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(review_source, review_target)
RESULT_DIR.mkdir(exist_ok=True)
os.environ["CW2_RESULT_DIR"] = str(RESULT_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
print("Mode:", MODE)
print("Code:", CODE_DIR)
print("Results:", RESULT_DIR)


### 内嵌的重复图片人工复核记录

这份压缩 CSV 来自原始小组审计，用于后面的重复照片敏感性分析。代码先验 SHA-256，再写入结果目录；不改变图片划分或训练标签。

In [ ]:
# Restore the same visually reviewed overlap decisions used in the local experiment.
import base64
import hashlib
import zlib
REVIEW_SHA256 = 'a7752e4f46d4b6596ade089434068fc0f81acc2c6e9c49275b95dfbf089cb267'
review_encoded = (
    'eNrtfVuPnDmS3fsC+x8K+zQDFNYMMi4M7JthwE/+DQKv3fKoJUFS7+wY/vE+kSVVldSVt4WxpQS61VDdMuujeBgR55DBiE/r39+uv795+36u/7h/+1v7Zb1p958/vnv7BR/Hu/b5c3z9a8ui+OThBf3rC/rXF/RvL+j389f2+dc38+3nL+39WPcfv/9ytPfz7Wxf1pvPH37/hG+s3/qa8+37X958/v3jxw+fvtx/ehjP51/X+nI/13j7+e2H9/f//vbz7+3dG/xsLvyif/6ndN8+fnyHobxr429vPn348kb/9X9//OX+y6f29v3Dz+7+793Dx8Nr7vCa+7rK1lIWlVkMI+5VijfOLc2+0/TS91DLRpV7azJK59L65DVIU9b1h4cS88Nj1+cvR5+6RIRr89HGsFSattJ4ypQeX44xjW21TZZo+eaUydWFpVQxIZr36f5hXvHJfvvL75/W5//2frVPb+bvgGHE'
    'dH6dtZT+9eP7X+4/t9/Wm4+/fvjy4f5f/lf7Mn7FBN+9WzGTdx/bly/r0/s7IHH3eaz36+4dPv2tffrb53+7+/D+3T/uxqcPH+/vxod3gCg+/vYRDzygcPfh092X9R9f7j78+/r0rv3jbr7de336/K//8s//RPe9vcefr5PzDv+gN5+/fFrtb2/+e+CbUirPEHp4MSbr6ycPsxXvunt41/1YY9nS3VUk0yJvtak1a6ssXbmt2lia5d5qH5jYSmJJF5POPsucZ8djD+P59/bugtFgCWSsD6tmVnXWItl66raX1KlplMp1lgaoa8/cuGLoJfnIhBXEMm4Cw3x2zrJfhWHWNIfy7Elq251lJuW0SAawZcxic88M8HKSlHxh2tooe1RJRab34+P57pGf3/7Svnz4W7v7C+e/Ptnj+fG1tnP3KuKp85xzrwEoB7dUenIhFc595LZhv1h1ZjTy0LEZK0BgqjeBajmL'
    'quhVqM7M7L6Kbu/cVtax8tp1Ls41G9YIphPTVY1KFxoueRDmsvUAvqR9Naoif73CVC1xZZIqW41X561bOhUxpb11NPKqAE+JvXtbac/sxbNMrEnxlW8CVL56Ev2vf4J8YyDLt0kcmKVPb9b7eSAd3+IXnQD08I47vOPAQPbomXa1Ude2WoCRtta1wd/tssx9pG075czFelHviaSMYhZw4h0nx0F83OX+MAyE8y1NddFYs6U2GsKopw3W1RJVwpfDVyLK2lvpzQXjSlWd4KSW3gRoej/efvn0++c3ILt/W5/eZMnPcHr4GSbo6ycPL7oXaSPFP78ssIc2eXhNU3YFMy1gia0pIuTA4sZ8q4B0IHI6uQtIh871wzPFH03pyANtdWt7TlEvWncQFZ9rbnYdGegriLFbzXsRhlW0TOXCXruS+AQTvhwKejUo7IdpKXQBFHMuxYxiHhqouMFvdO8lzzRrwjelgNJJ'
    'rF8m8HNjyWngZ11gYxtW9CP8VZ7M48gjV+3gizYWL6zyotAIRM4OqlSyuHEaRmCWG9YDTycKQimiE3Sk4Gu+CTDq/ccPX+Dr32Bg7/6BuPD2l1+/vPnlw4dfoKjgzZ4zy4dXYp6+fnJ4y93DW+5rLXAJHf/8VupKo1WPqam5Z8DBeeYNpDZCE8SAtz46zd7w3WbJSe3+y4ffjo4jqz3B9fBCDOPrJ98Ng9Ji/MZGu6U1MYJSBMR1sRIeLPBzK7FIF8VAFqIPmO2E7UDGpFHTbYDm30B7F+P5OlflBFLxum8ztDokM6g+pS4ZBlS2b7c8VxlLGtzKHql2Tqk6wJJZl69mtUasygjfLz37cb1wfcLp1CAMFF41E4JOq7DWXSt12lMF0YV7kQpLVki7CuFR12rT5+iz7h4hauybgInS/ecP/+irvQe7GnjE2/bu24w9RGn8+5+h9vXFmLFvnz2+7XHeCBbEYx+W'
    'b96I1ZBrkNjcpA9GIFcIu9jHqFRzBsHwMjGdHaRhgGDNcwNK8swvXjAe3fCBHTZMcw0eBvoQREHB5TZU5YxgCe4wBQFzcbexoeYKwueWpLZvw9yIjk/b//y28nO+Esm2EeFB57hM8wyW5b3mmrTD23mbGttMmLBW5gCZ8LmnLbJNQHgBzrNIfltalyJJpYF39tp6UpqgPNqHWQXj3PEpc82pQGcnkJLRQQ1rGKkW0NVuV22HvSKS+TySma9EEgF/Y7H3ySVXLHpIL127GyhIYObwqQuxrsAatCZYw+YeKkmVMlSPn0WS/SokC4OwYN2AvuvgugkSUaEbtzeMBfKwQ4G15soj82zQZ00kVeLFCJ1Xkcj8ekiWx2mbH/7+/h9vfnv7bmJI33Y1T/vVw1vuHt5yT7EbmGl18DoBzYQ0ACMp4C8N9lcbQW2Tcufu1FMHxtRSHXBkeF3TcXIgfNKffjcObauBxmPw'
    'thUOHIqZedKosed6ME5D0G5gM31AdAwanEA7ERRzt9rbbeDGJ6er2OW4jQIe3stcXR0OCP60Sw2lhhU+Jgjmwh8ABNSc5ph5MltwmeYJytVODoTkUaudGQYmv4OlLC6eoEO0wlv6GhOyREZesD4476q1NdhgXj4wGAWZGURUG/NtwCanZouELocNbilDV7A4dDR07sCKRhSMPaIlCl81wM/B05tnBQk17hmKodoYHFsU+yRsuVxsbkzMcMDdGA4QSgWO3yG3N/5qoDISFp9jJ5y0zTKh72eDxyxuptY93QZu+vJ0PdEWv8JRjpUmlEBwdEQ0Z9KO9YxA1hBz6pzWYcCyaoP0BketEFR7Bw01hEitpy2/Xowc6KS1tVJhQ8yCxiTow77JaW0yBDOIT/zD6oQKX26998SU8irsvex5Ixb35J/2pw+/vFn/WF/3az8+bfJlPolevPEOb/y6R4s33pMRyHoiBDco'
    '4g4ZAXkF/bWSrrVlTEcsKawZzqzlWt0RE0EakoERSvYLBuUnQ94LY7I6EoJww4pRMCbqTCO7QkpkAXrsVB0BESPEitIKb89T1ioVSsi70G3gWS+YuuJX49lKhJZaYQHMoxpVOFCDWjbzAb7Z+0ZoxMQF45PtsTFSZwG1cNjFBXiSXItnA7/N+B+RkRb4Zx4EOpyg7aHdawYNnXFCUJwZWmePnUpRBM2CuLBavsazltfD85KpK3ItnhDrvIwcgQ4mmBwxacPdZpgpdMWuTjmp7zGhuxg/0MhQWIiLunPlvk8N6tHpi16J6NRhwGq2DNrZsHz6AvtF/M7brULs5+GpOqQ9GGtaWnkNUC+43FHZr9oKfT1Ec7oE0XotorVtr3lmG5ARAp+adp4wgBbhyRLb1JQT5NqUkZZPlS4+JuVmZcr0ixDleq3PjXUGiyyUVou9PkCpIyvtUXZqIKarOggXxgvCI6ViLYLH'
    'Sh7cWt96G4jSBYhWvRbR2UFLpcwkAgoJBpsRsRrwm4taa9KVKphHLU0hHWUYeC3MBvobbhBK/JJAQCfkxwtDAovuwl7hbhkWOaCJVKvkCJQbTE1dChloWK9CiPcVNA5sjSPdqlDvtwHn0+7Np98/f/mB8788V/HCezUt5gUqzARsJs6mm2VMCdfVQRu3JEi46aOCDE3ROD5aiXqT1RtUwIuPPR0WD88FJNDvZRtZWgvLZCjWAEnnkhs+35lBwBhc1nT3PEoG71perRUw6j5uA5Xy0vQQnQ59h/lJc6YiCxZiLY2Eyai5rV0kLTHLCZMHOg9dCDvK4oVkOWmmHrk+q9FLD84i55ZDp9whWyZnKhu01hbibIGSadCZo1Tx6gOEBUo+NM2Cq55DsBA6LJqvOnx4RVj4xRM1uKtnuJw8SkNYkAFOySnN7NBZhCWcZ+xNOxje4Ugt50iHVFgN/h+LiyabDY5Fub74'
    'fLZHdE6f48mGtYJKQHUXziPDdSnID6gQNZoLpjl4JV5zapMZ54hpwQtDn3sluYZx8OthJCdPPZPppVhlW3s3qyJ9Lm8NUQYGBBWQSmfrNtcIfTxHg8Ulg9rOXkC7rSG6W0mnT4G/sosA7eQZcNPadSxnPAFWBNBsUYLILCV5tkpJEJxqqo0qlIBBEHitMWhORRvdBmh6//dfV/vybLv+eR7VXx5Sq8avbe+/vnn2xd1f9Hmu1+F3YB4fPj5t4j9Lp7p7+F13D7/rPu2luVWBAgZiLXGuE9q4Qzd7hnQsBqo5KG2Iwd3hW0M2x9dgAhVC+T877GxPKWDXjxruA167cigFaE6og0XTMDifUBS1dSwDHjOcjalUbrQnNYWTKfA4tG9jTdjXycVf881nkLHHgwg/inm89i5ee7/ylF7cIII72CQMtkTKqxQFe9BwgzCSyLviBUOCQU9igXxuJY1hZR15OBE9UZRj'
    'z4a6RMgDA9p7cve5O9QlXGiGSikC1roQAcSXzS275Iqxro4ny+5qe6TbwKcemaJMfAE+we13K5GvkEtGqDNZk7kNc9+zEGdMn4BPBlTQ4dIGgdLJwqf4n//w8G+6rSQ/j1DeZa4VuzWH3x/qC45TGpbBTNsdOLSce9dIRav4uQrhNV61x5mt3gZC/nWSDk7pOc/+jkh+P0cHr3OgdJRrX9v7mnFkDYh09VVgGwzGQIkxaeFvSOGMcuxQtJZVIbIhz4Yl/cPDn5R1OobQ09MHmLtAGmdYBK2kBV4s7j0AJZq12a4y5vZSdabKlAYpbKnbDnMC+b0JhEASvk7Shw+f15vPv/3+BBEdCRCHl97FS+9t2FTrtHlJkb5F16DADMR+D10zZW46ClOkO/JSn7QTzcTCDfZ29OFuRwF6erpa7hO0pwMXcNiwlhYnPq3NhiDEMJUGe7U5uHeyUjKXjjWSFciuq7Yk5fUQ'
    'omOTRG7HjehpljITFNEMty4Ce+HZBCF5zAopnJt10HqwR8Z6B6EcSxMZVFMkpG6p1I5ilNL5BQI9N2Rkmk5UFQoMplkXOLAT2GHbrefWW/GVUoPPw0AjxbBTg6McU+U2IMrH5ih/naMzEOluBD6XdqFdS4LgAQmLVDnBJ/CCc5urxM2dnON+zJhl5kng3Rncf7wA0aOnq3SBIY21E09l3rHHFFuUWir86uob3C0l7fgMPwcplbFB6p0lksa7G2UQvdtAqRxFKdc/UfpZUOIT05Qucngr980lzj3hXziXRrJESsgSm7YUsRleJzG83ByUxxAOPwc9Mx3+iv90eGdBkq9z9PHD3+f69HTq397O3xGY7Lg9fX3HY97Ikulae4qzTSoD2gTUG0J3dcrOxbI2HhBRSxI77A9AbtM9OszQvZ0cx7f0zT8C9sMoQAh8WobwWsTMpZJDBZQCCW4glfh3O546SyaCCLfs'
    'a4B0eIr8o5SNbgM0PTJZXzfc6VLM4FdyHCZZGam3zKO5gw0DopxHANcLtVQgcqFs6zqQszj46nPsTEYnh+F6GWQVopYg1kCCaurMxiCVbHmnvi1B5aaE54Fu9ta7Kf55IPIrFLiCKK5rINPXg8xOzRV9lw95cracoWAKCeZ/RvhY2dhaH7sdyKDMLNZKiVN9eCwz4DezdTinLmzpCGTf/DKlo+Hrh3FwbzYdAhlxzCe84Wrs1iKhvOwOPyA2vHnqCtcMPQc5ZSVDStAqTa860XpF1OpJ1LJdihp8jCKOGIh571CjcQmKwNatQlINaJwNsZkQ3hBFYme4c519ZRkaZ1CUTtt7uRSzmQlqMPfS96pjQDy3oho6oQ0oXojxFXsn3kpezStMUTG+7nVPuNF2G5j5SczKxZZWGBoqUZ3iDtSsxEa7xxW2KqJpDU8jVZbmyxcpCCM+VbC23SO/e5wchl0az1YtYB8c'
    '23xp5YAMzjicZRxSOiGW1qxdwGMjf7x6bD5RBQtJq0rWm4CM02mvBGZ8cUSrRVNcbbEiNenocYZRXDE/POLmYQJNiZzEmKi4fw13lNfMqzhoZNl/GtrlqH2LJZ/Xxy8fPr2NW+sf8OjH1PpyFLRv77h7eMe9QXf1qhmsEXE+UghLbSVSDbeA53epUcUE7GDvKXVFxks2b5Bq7K7bjwzk6wHZ49P+IsdOSn4cECJopylNEd5yZh6zz12lzArrtmIrcnAQZ2F/ZDJXmgQLzYBwQCzQug0A85F5e/jwrC7IBRDusUeGc7SWbC1E/KmzKKYGCjcNS2PHaTCsTnO8Im3oKvfeS9zrgAXeMoT2ehCWMxBauRxC4jij5BrZMFEvqG/Ne+AbPQnXJGBwSk0c34X0VYdsqFG3ojWhBKZ+ZihyMXRlK7R0Xwx2lAo4Pr6curdJaQIH4OxrblsDrxm9Dkm202hc3BAVr1Le'
    'rwjdt92Jz1/Wb9+dqCQ7Nk945cOJRjD5EQRAFuJHMPs52ZMIxRV4KTIjZabB9mhb5KjujgmsHvkzabodeXROeizCPT3bsAKkp4Y4lkBPw8ryFij5hO/hySb4F8gYFYY2u2b4gQ6Z30Y3sdr1NtCRY+jQibD2OEeQz6TmUGotQwxF/mYBZxs1WaPteY6cUz0c7FoCbpxACTomFN4IVMFec2ncBDz6M8Lzp/k84nN0BZueXcE5ChFx5P5t17gLiGDUEZtkVfCGXUufbQIXG1vXqmqDk2Z8r3sq9JhL8OzRXzcRUzkPDxW2niG6WiRSL48t0N5SzwJeivDjGB2en6xOLjIgEavFrkuK7fm16TbgqUfgOSWPn7mYqKqTDts+6qNIjg26Io4F2kEDk7de924KstfJQK0QqxciM6iEA8t21HYvMJ+yJS6SmsRkQzV1m9Jpicd5ccGA0s6ZZq1a1rChBUMSnVhEM+V+'
    'VQGR+nr4+DF8ThwUP83RUoP1+HCw7IpoYKlbnJGM0T02Eii2x8PPxMaFwWx6ZI2WZhM0r412Kwb0egBJ+skA+q9zrjcBD/1hhp4uV+U/+dvrA5R/ToB+Ngb3igg97ix8+fT24/qeJKQTCMWrH+apRsKjT2MaJa7QNC08okisaym1dqBTjKRn50Wjx/2FKSJLUl62ff2nE6Llr8dAvCgjOqoYRKXHNEXb6iOsSkEpUtRJqyAVDd65ZAG1iLtbewD8MrpgIQxN5Tbg5aPwZi0XwasueU3FP32kMpZj0nYR2KDP3JsgdKjVtVefExYncU0l1U617QLCSOuFATzW+MkniMbTCCKZFI5vJFsd/q+qcxyJdDjQgQ893MPwKnlsXiVqRmjc3rUoV9FTuSZ33V8PKbn/P7+PeNTbI8cS33H2b6/FhD1++uPBVo06Ul0ttTriULiDhPWolLhaIwWSy3KF10KogcuE24TZ'
    'gl0LtahLd3Y8z2qzXDAa0L+4YQBwsEjAgYimhJqYvs0FkW4qx+U8jHAlBW3sbC3BPgdHcirdBoh6FkS7DsTcKIjzHnUkjvOkwlzMK6hdHPxFqUfNS2eX3XcVATdIK/Zyl8TNnn5uPCk/WeAFw2HQRoMjT1GEyTGmxMPizL/B7M05qseUlW2AZMIWV9QR4JWw6kZRaus2ULRzs+ZXojhpgqJgymS3rJifbO65F40UKk91bso8VCokwHCFlVTfIOJFYJBUyrnxyHUoYm3Am1AirJSeam54XgfpT3mD2VRfsL9wEa1mrgsCb8CPp90RAw930m8DxXpm1ojTdSjCZSWaDaIuqrRNSnHFQEbcUm55WBRVbTCPZlENMpJJ15Sl3TDLMNORzo3n+S2UC4bTdFVCyMt9FVpYPzky5RunWqHDo84d11LWWKsXGtMVHpXSMLeCoKn5NlD0W0OxlmvCYlk2qM80sjaSKDdR'
    'cuXYF5h9K9ZEpA0Z3PyENwg1tL2BXyH6jrrE5k2AqD/bpN2qIdIrVm6nM7OWa77OEA3Bj/sGURgORY7pIZKiLJWtz6gw0PdicNVUEr7TwBt9g97P0OM2z/LTxFehOJdR3JCdfKi5EqGxlGKKb5NOlTznBBOjjVgZNc3IxNOKynWDm89xGyjmM7NW8p8o/vwolj/22rqgwVePOjnDpu1wk7s3i+xrBuXMU8bs1kE6weV9Dc0z8BuFXLlM2Rti+utDD1dnn5c1/v55h8uycTGXWkz4HiE4dQ2AQF7T6oOiklxfUPIUhbJXLk2g7h1Ard4PdUDu6RZg4K8zctjUeCzTQE+x7PuZeah6EneRqpXJpWKuR98bKFBLMiJLiqPbh0ZWPaRi2rTbgakgqizHTD3ddfnDFdlS/PwVWScAuSr0dEXA3DOwKLLI6ooCnMKHvjB5QGZigaw6KfJSQxdQhsXuq3F5mK71/ss3'
    'cP7Ht2/c/foBA7r7CCwO2z2/jy/4dfdPKP3bXTwoUHl36HN3mPLHri8f8ddvLbrnrfZ5Pc2+5hONXx7edPf1TfcJBC8d6qIvyGbNmTQKeHQonc6eLeEFUQ6Aw4EhXgumCgopNkpmydOubuNDV7XxyT1NP2ybRMIaYbEoRZmOIeCpcKTqI6ciVnKPfNfo/JB2uL4F17r6ug0j+rElDHzUuf4sMgr3CacyIyu6gAbChYdclT6DtU+o2NLiPAu0LBlpFOUaXaWvFWXef3iip7NNSGakqB6kMPtqacycbWvtWRH0YChqnPC9AXY5YF7b6qHUvu44jG77Giheryy7nq9mL1c2SkDI31yG7NTFuTdMIc3dZeeIP6WnMuEFGc4mTnGMq9uA00lCeUSJrrOdGyxfVZR96+TRGpWGKNhcDdKjxd6GSVRujNCI1UPSolr75moiZcIh4yeR3l1vA8enGqf904e/v/++Vu2p'
    'knaHlz+UsltjNEm5dbj9xaBvOfeo6VQjLZN2TvAzPsH9EBW2QXypksDUWFPVVtaJITy3t5NjAKOkUgblKtHOgrLCu61oTTqiX2jcRoIshzgvMydYIdZSaW3A+Jm4idwGWn6mNnS6ohg7PF6LOvW1IVZk31G0YSwrsCoeDKLXvRlUMha5jyS5aIMlWOsIdjBU/2mK6N8CcpZ+xhKSF1WetSvrlCpcIfy0d44WgiEXCux4ah8ttcIY6vA24UVF2irRgqpFMam21mgGun8beN5endI/ET2N6FPN0N8+fG5vx2P9CD1hlQ8vBXeZDtodHblAFxywgIZjOkbMDC/JKmuRjpSHNGZvY0Qd91lnofG8pvd3j34UKC8j9fXZ4LIj1DPzJi6r2SHBkeBNBRGulsEGJV9GFP+PGA1N6dDcNXOhvGu+Ap3Xq8VuL5YsTcTnS5Y2M0QLthFlMjSOzmZE/S1doo+F9hX3x3fd'
    'iCoVP5eWy+gzbgMl4Z5efLDx2VKyXjp0MMLWpJnXSu1wFQVaMe6n8EpYBJCPo8yZIULIsYbyAofhnHKI/tvAhU/Wwyxsl9bDlOJDYva5b0w9GNpwjxpGcCQaFT9a9rTUyRmOpsHnRbGGQ4vCClanp+tyil/YjTDKSEXH3jZnj4uZYD7D6ky2LMPRpS2bMDTDtwshuEYlxzlBO8Ev6ya/DdTkZ0Pt8YbYU7x6qqmajmP3eE/sWS8DyQJuOPuh8WvBs0qpe0cTjZrhgvsaDEaUlGlDItQouuOcs2iRIancBoKnZo6MT+D3wpSBLQphWUuUUt4WpdXgpkC/G6hGx1xtOMXl5nlpaRmxLtFIs40OKr4yfRvLj3l5MRirx9H7MffuAN/M0Yt3jmgmbzsPjx22nB2syNvh7MAJgrJxgeeeUQ/JaPccleCcKN0GfC9UH/1WXkcvKG952LTrueuaVLnEBqMqpK9GQh3I'
    'RoUYjraNtRKif9SIi24+vUcBjjTGOPb0on4kG/np2bQPvdaSQLZlUiw296iaQ9FQcqi1Werc2luzWVO0c+UyoczHaLC2cSMesh4tFJXlkjpVFOmfnOqaPmKxSm/kQSzgETtFgxWP7cXgHNuY0iyuYAN7R+mh5afqiekFtRuT9wXnG0WXtm4KJ5dIJ/wyzHsdWgd6I2CzomUQbNqi1YVHSn9EMbsCpNcrmG5+ulJVvbhGRArVQjY5x+0LX9WbAh1tE+6IZMRB5Zhz7gbUigOyBk1WKIo0h8Y+XanqaGmxH9NNPIoZREZlqiCndaS4KjMqb5AP+OiKBRWVREpFVNs0O0XL0R7YYfgICbcAWk2na0QUuvh2um7uEn9GJUyXIBLBvACJR7cBT2kuxHMbM3dwtZ7WKn04hFhMl/M+czs9p4tvpyu1SPqm3RZMSHVnIylJ+uTK3QUWRj5KHCNjDUXWCHWJ0Kk0uK3b'
    'AI7OzJfq5YUFGkiFLe59NEsjc6O0hjUQjuhwagZQYZP7UFlPIZrwdRQJBqC51fxYhOnYUEgvhQ6/f6zIPmVCFBWbrRE4EOIkRpDiEhYhzDU5UH82K75yZyFAu2cadBvQnSvEUK8o6xGl4XoUX4P1gaElhLWGtdxn1XJIocAKZ5Axld4jc3vLIDAPz2CXCD/jijscL+R1XH95wxD1gFWr8N8WVYrhTndRyVFRs1RYqs3CSceYPWWsvBksN+1mC+Rm5ttAuBy7fWiX3H5P0fxNE1zRzg5PirCHeIeYYnFlypU33GizqLuTZ/QnnuJRFbv1FJecyvGrW0YX3A9lwBNnEeq9KuIwyVpaV58NfnJm7YM21tBE2KOVDi0hIedG4wX2228k7r14veaBIbDoRfdror4oKyJGRZh0BJthlKCqjSc0gHFkIxSHUzU4NfCY3CGRVuQuSRsl2/ERUE6XXK/pCF/JwP03qcrh'
    'up1PxDsaHvpCo68sXEGrYLlcN4R1lPuztDc+bLtmJ/j1yqZXOXoRKiW67CLUqOZk+AuEBAwttv4omvLEriN7WZr3YsjVNYKoww+Br3Ck0ucssvT4AGo+el306fEwzhBbXTNCVIcMazJ77DfHX3HnEVxf48KF0KKx3SdCrw3GemIrorcB09kLNCbX5SdCd2mo2uhhPHItOrSVqDY6Wtoe5cVKO1xdAwnZ66DwYjpbnLNFAvi5rNd0XeZy3s4dMWhKac5wuy7RLZejM2EN1qupBPPti1XZbeO1Fd8pukbvNwLiU2LU7+/Hr/948+XDx8dsfTqRnvXw8ju8/B5eyCNVWlPrXqN/RoGDZF4pyj9HZ3hB+NC8egmJoAwzJIfOnpIL235hBI/FT8WPJ2E9jYCilcXWNUercMBjLG4pihojhibYuTGPDtoBKWegj3WBV3aNbuhxVUvqfb4FqOpRqL7fMP4TqleH6se0'
    'tfzdptWRxLVFkKPgEA4lq1LXhmKFxNa5d89c4GNSa9xs9rlYWGKDMSXdbQeVNv7hmcXO5eaxmEFJCEeRmSgMvrT5Ivx++F1QwQ220+KyWYF8hgtesU8VVIMU9IbTTWDh6cd5cT+PRdxqLSDFVeoYubRUIFrAzDOoRIKOylFRh/JkcOMyCLDx1KQgzdGUq7QfnknJzuYt2m6jIgKWCmZJoC2cJgBuqxBlop2ZJ5dUWQ/Updsy7amvWiNNHuTycjRer066048ppJQvQMONKtbfzhKdmtaIs6ItS3hFSlSBWtSmEyplRHc0+LQJD9N7rdTAsPwPlnE2bRU8rYNhF6nREHAuH0tm7bRgybS3EKhIU65Rj72NKAATl0U5jkFA+ne5DTDyiZzNr3unpVyXRQo+y3W2Ep3JYRTk4ogtvufIgw+9BXhb1Ihd4EkCuTkUSIrsbZrAqF7uV5vPnvZLyQpqGMdQUf9qBO3O'
    'nMG7o8dYYXwj9uOhzVbZoOWVpsP7wZYjezzLbQBWTib9qVyefchFJmLxoReE6VKFQz9kTAMLlj4iQ2Vjae9sVpshukMrqbU4poD7G2cSIclPJO18NxDaqgXPKF3A7Dka9nEVYoNfk9jPomgbN6NrKpQ1iElcdUBgGtMmtXUjwF3SwZqubpSOOZNoem80E5hd5ACD0anuqFtfs2EqC4LW2l2UA9Ame5iOqQRPlcZl6Wp8ZVZkg96GKfLyaEOrY08qxc1LjvvqqYJ80qxesdiwuGJ/y3cqC4gLxO+NACovp4x9f8X5aM7Yth4VGKo6QslCdMEMxEE+5ipHDw0tK7umlfBj0DzaAgKmIu4N5ILTkYfTJflq0aFy4sG+d5pwgnVDO0d7CvjqFRtIDjOscA/qfR04edcqq0jq0OJz3wY++nIH8nI+X80ne0vRGsJLJB1ZgWG1FC2pNpURxNgHPOD0yCMbu1AXaJja'
    'EkgYSPmLDy7nIxhsV/J2sYoQahMxFBKpLe3RqQcBC26xS3Sl0MiIIkQ3mxuvjZ4jmds1lOP1Cpy7vZj5lOtlPa41w23ljHVY48SPqmH+W5TFHhZXxgV0cPYd6emk6XAwAsKIIO9r5irj5QbfVi5MTpsLkDulSM2vkX9bG1Rx6RzlNrtGa57uEGMO3tNKHFyTiS0I7BnlL/02IKrHslNY6wW5MZCTSr7zqjCR3CP1lnOuW7e13nXtZdBOChOqkPazF4geGFKLqIUgcvzp5YLevNPKIZkqaouWuBo7S2zEj8ih0gL3PKIvj88xI/dUOihFLR7NaoHPyus2EPJjvXn1kt7JErveNY5SFQo3cpWgNZdFngpUZtbsTnEFOEXK18CShsuLEnC9N5DAfaxxc6ELOvO2yhBLQDuS03dvuiCk4rqkRzG3uiosi8AVbEJHgbSApUcrdoPQXr2t27AgSscbv9pFXfYOJTAy'
    'FmqLy6OQt0w2xQuok8cGA+g5r7qsOpgVA6YelX5nodXGAt173S57t4ERnU5ecr64/xe8Ci2f2SEuEXKkWY2thxWX8ifW8oZHVIQtMPy9y4K+Eoo0ptwbONg4nbzEfmGHm+wbDGGXERef9xiVclxGt7VKqbYLFg+YS+wc2pZeUhR4qGUhniYHH803Als+VshV5YKD9j56SzzwXwjaZUGz6ySKU1twrlyktswg3VnaSj7BuOAIjYwUbqrl163DfDlC9ooIlZ8Mof+6Qsy3gc+JRAg6CdHTAbczZirtHl06gQJW8zYeWrBms1EU9gXhBr2asT/RIVlBzwsCvmTIyTnn8REUsksSISz1HUkxnWx5JMVbOoCkUfl1gXTC1hC+vHbV2aMTTU0g6LH5wUu53AhUcnXZivxdd6HzhSt2KtqUJCF8YeZKy2CDKtwbz7j2n+ro4GUgX8lhc3MHZ14VviqvDGV1fICPnjFf'
    'UUdDVN0sE4Mf1sWWZjGTFIeSc+06h3bwEE5xdzrHPfNI75bpcTIFNyD35SZw1SOlUb47JjxTEQWq17SuQ550HbZb5JRP3tw4jygOVEvdKSo+42cVujhMlkbUfQK1lHFkCN8qqr2E1w8j4NWYdM20QEhnw6KBNO8rKqVnBpBjbZal0BhRUyhFKdlkSRkufs9O80bAsh8P8/SCA8SVhWuNenY1uLvBNcEPZm6zRVk7xHae5CBeSZtZ20u2ItxkaByGBfx4aJmfWdGxuidpgpH32HIVGX1FVgDgdnzBbeylSYtEVfTDxiLoqh5UMjNFWdFZbgSN+uNhXr3gAJHg4mJzKEWirCAUlShOveFREDBqK7VL2XBqyVOLy/qjx3njztA7eAfbT41GfUU0jtycpu9AOb4Vvch3P5TWYobH0p2XpLiwnSIFNZrwwp9hRuBP2vYaHVS7D4lmgB0i6siNcbLzN8aHwB+tqRvi'
    'hxYkUI3ol+I6jkIy5Yq1sGuHnsuRdDs8NG4RjmI0WB7rNvCh9PJ2K6VLL5i2YrM0ktLj5g3sJorKcN2wEsmEJYzom6LDO68NJrwaiDBU79hxy9pVTlyTzFyuuliq1V3izskqKzYONUMfyOpJIhFPCg0H79ylunJLUyVZg1WvCj0+KsuNQEZHN288X7J35A2+LHofTBqQL8wDFE/KmtXbUF+QLMvA2q3V6HVbIUHnTiONXK3DLR3fOzp6DeDZ02uZoIxgalC6w6eAgwyOLKLRYOWIdAPGtmHHlTpoJckwYbCFacbV9Fbs6nizlEKXNEuJpFbQsLqaF0PMwfLdiohEUYYHpjNbEXhAxAraGfTW17CaoXTNPArWvu4+xG1gVH46jP7rdiJuA6ETOxFF/KKdCBmjKRPvheWaaXMlsoWP20uVyM+GkB0p2tZihbNtq42xmHV3siH19XciLofKXxEq+VEEnc0uzXWC'
    '3JUd1SmiG1bF3Md1ecqR2bh3ZCFA0Gdf+NpAu326eBSxmOBaPf/wQD1f+bEkWAatRGylaov2FniGLc3msOUa10A8LxDLjFDkq0Vn6RnVlQYJdb/nm4BCT+de0ZmiPN9lXxXE3sgooLR6A7FNo4RZgEPMuA4Wd4pixRJUU1w7jAu8OWpPZZhRkz5P5qM9Lz1xZhyVFNbCmoTmMOAzcwMzya34CPisRGdr5mAOJfArSUUjZjW8r94KdHaqUsdj+ReSE+z8pZodILsgWdEzXaLdk0JJ9oloLoMquB0XG5ZJrHffWuL4bkX2n0qtzCW9dqHia7H7/1yomKieLl5U8qVqKQpwhHI87I+DHBgYQ9mFO7h7nK9HsnKO3J3StA2eqWOV57L6GvgZ7ZPjyE6XFlFiOLUZCdOkhThaZMwBnSatIvb1BuGE8cVt3JTiUje8cRWHGIBFzX0zjvCnTIOAirogDSKqQnrl4Qwz'
    'iJuiWWXbSIwQJg2kblCP6/d7MUmKI9sE+Y0FAwGVM98GQDkdOdr+Tk+eLcvhmCDTXNRyjx4AiBB1R+tIBAPbpdcE7c81L9jb2BC55XDm7pFnVOmxaeHL/UMsXXi8jt+E1QJpBMPtNZrFwuMBnZaKgXAweB/+7MqHTm29McPIufvqvcDL5sshy+kVIXvWOuSFkE7p3JXO76J628URfuJSWRSQnjzXhktaJY4HdsHqjmMA8Iwd92oIwQNqZsIuVzQ1KOvkWNKZ65w/pLvbIQ05xfVR4khxyZMUfFA7SYWr7jM27KEfolzPsqi02TKAjrq3Xm4EvXwPOnA8kNXnNzofXokZ+/rJdxEEeKhHWavlHIVSNPZoHGGspxUXkTbcU5SU4h7753EltoFhOOywcakmp8dBz4jhyWFsiWv2OY60emfE1iIapXjGQOTcxjsGR012lh3X4C1H2TICZd1BINd9NEf++ANuh68f'
    'Wy29JlzlB43zxDjsghMmy23lqCk1ELNyj5L6BZjVXDt0ZzGGm4waElGDRxHmKtg8MQQzwr3X+l9/XVDv808MBp9cs4WfWmieXLKgJXm2BRMJh5PU5x4NdC92cbhpZKk4L+EtK7W4tdEE8apriqqoWdtpy3m+B3tyGBDAu44dl0R5jj7qFBDBSG0vDocGs6rwwW3nVP1h7wROt+8W235xZHVff2qw5JKrK9Ae111dgbaEFyvQvXGzJshX2gmEb6vVKCrliEl7lMMZ3Q5bm3Hnb4DBj8N9sksu+eQriz8Td9IeedQDprzxBUgoxAWi6qwEHRdFyVIUTpK8VrdaQU5mWdHauUCP3Ati188LpJ7oOZzosq7WwyaWd4gshU3tSpojoxhcAhPGcIIucKkdISMMEDZBDbFksuTo51n28WofzhdU+4hacb1IUxg7+ETeE5O/rMP+SofdT5E2vfZoXG0uMPlVZiydARDx'
    'CfziN4ToJ0ToD4fftZ4PToNLtCkYBcxdasSJwg0cITQw1cgS6ZLrHBOUIRr3gGmloodZWY2n/yHlopzd/VNyRchPeQAQSANM+8BER5+X2KEXybHhaGP2Fv12WaEm4jYJ9HCqWD2wtLOWQq+Iw48BO393NffYJiyme9GO9rFx06vuHY0JUi41dGSBhxwsIBDgUr3rwByyrMWBks79Kmkoij8/Lwp+ScVkSVfW/51TKoCoJD4aqLVi/icULYFcR1PmaWy+Y3Op1olP4zpR6gniF8Hoqf7va28b/dTQlSO5EH7p5h6J7Di0bkSdI8euxQ3JAtrUZ9PIRphmmrWXSKWE4szwf5mqS90Ep/QzpULYT40UHY3IpPkiStAXZJBE6VE4sz7icvlebWC19jmio310LsSiVvFK1jCbkClTtXhj2NcpVkJcLzkXrAvutTJ4CHTaSGYEaxkMNbYzSezC94KQl2bpsPGawriz'
    'gmZG3Wab/RlE9BNClG/GDz5v4nH6cmmUyBsJVlvV2bTH8qiUZ/XlI0eXUoumIuYNOhgCIE44clRAhLVluBdKZ60qvyJk5WSYoO9K652+BjsnwgPWd/FJVvoueeS6QHOD5o2oCAxC0YygJQ/1JBDiC4RmpRH1SO1n2Rwi+qnx4v9EpdbjF3QvKdU6EgTmtGwcyT7RJAJsLvVJlKIDUpTiQADMDr2p5nkmXZFM1sDyvbbHa4eXDPjZF3d/oa+Vnv+TFWY7eH21CtHXYJtbQOcHtB1cOgR79AtE+MsaR9swVvgJaIKGhQi1YSW6N/3sC0Hux4dP79+8//Dpy694wsOs/mi++p0awOuDmceHb297mMBvtkMenfooR8l7WExjRB4qM4omTSBbNa61QzC7RiqN25qQVm32aMTiVC4Y0fP9+bPjiWp0DiIDw53wrDwNIbI2ydEyKBIUO0kWiqPSPmQ0GduTDQgH+BUq'
    'dh8bVj8vgnosXy/LRZlo26JHffS4bdFdosSNl8xrwqml2PaZEHZDy6rk5D56j8YDWd179no8WxDi/ny2ILSZiAKb2rf0BeYCES0FEi645dDcWmeAwztVCM3IzICME617ThvWfnJoXi5IQV4uzpAGzxnMg6O+rPPetCCtp7cJFgG5wBF5onxR3NaBFcH7MHifzdZTCK6Xn1/0QtbinX2tkoA4DNOh0+DfMoE2URRt6s0i/2pGUX6NRu5RkYIseqI1bUIT4JyDp7wiPC9PD9tl9UJINjxZWiqzRTGibNTVoeIlE0LAarKiFRgsaWIyJlxPS3Gmm0v0TxX+E5tT2PjNFbX61h312g6MWCx4ShrGUc4kl71L3IRIUQdm6+4WSfZNIEMQO0ExhSmuRFgUZIgrePf+MwPJfyiRmS/YWmwsSozw23RAIRGigPXo0W4V2rZb0d0iFUpk5Kgn12f8vzyucUbdlR+3FkXO'
    'bvGOraprbQFB2Bwl2Eu0Nt6aBaQ1R+O4ndusrW/2Hq06KwwuzC5FH1YJovcz4/CH4ph8HobwFR6tn/eEFykxFTCDWiYkLKlWhr/JUqKCr8vwmStvAwYDK3dExsBrwCA/MQz/Dw1xC1I='
)
review_bytes = zlib.decompress(base64.b64decode(review_encoded))
if hashlib.sha256(review_bytes).hexdigest() != REVIEW_SHA256:
    raise ValueError("Embedded visual-review checksum mismatch")
review_path = RESULT_DIR / "tables" / "near_duplicate_review.csv"
review_path.parent.mkdir(parents=True, exist_ok=True)
if review_path.exists():
    if review_path.read_bytes() != review_bytes:
        raise ValueError("Existing visual-review CSV differs from the notebook's record")
else:
    review_path.write_bytes(review_bytes)
print("Visual-review record ready:", review_path)

### Dependencies and hardware

The training code needs Python, PyTorch, torchvision, NumPy, Pillow, and Matplotlib. These are normally preinstalled in recent Colab/Kaggle images. The cell installs only missing packages; restart the notebook runtime if pip requests it. The model uses the official torchvision ResNet18_Weights.IMAGENET1K_V1 parameters. The full run records its actual software versions and device in result/environment_info.json.

**运行检查：**先看下面输出的 `CUDA available`。`True` 表示会使用 GPU；若为 `False`，仍能运行，但训练会明显变慢。

In [ ]:
required = {"torch": "torch", "torchvision": "torchvision",
            "numpy": "numpy", "PIL": "Pillow", "matplotlib": "matplotlib"}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import numpy as np
import torch
import torchvision
import matplotlib
from PIL import Image
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

### Official pretrained ResNet-18 weights

The full mode loads torchvision ResNet18_Weights.IMAGENET1K_V1. If the weights are missing, this cell downloads the official file from download.pytorch.org into result/checkpoints and verifies its full SHA-256 before any training. Demo mode uses the saved coursework checkpoint and does not need this upstream file.

**为什么需要这一步：**三个 CNN 实验使用同一份 ImageNet 预训练 ResNet-18 权重作为起点，以便公平比较。程序会检查文件的 SHA-256，防止误用另一版权重。

In [ ]:
import shutil
import urllib.request

WEIGHTS_URL = "https://download.pytorch.org/models/resnet18-f37072fd.pth"
WEIGHTS_SHA256 = "f37072fd47e89c5e827621c5baffa7500819f7896bbacec160b1a16c560e07ec"
weight_path = RESULT_DIR / "checkpoints" / "imagenet_resnet18_f37072fd.pth"
weight_path.parent.mkdir(parents=True, exist_ok=True)
if MODE == "full":
    if not weight_path.is_file():
        temporary = weight_path.with_suffix(".download")
        print("Downloading official ImageNet ResNet-18 weights...")
        with urllib.request.urlopen(WEIGHTS_URL, timeout=60) as remote, temporary.open("wb") as local:
            shutil.copyfileobj(remote, local)
        if hashlib.sha256(temporary.read_bytes()).hexdigest() != WEIGHTS_SHA256:
            temporary.unlink()
            raise ValueError("Downloaded pretrained weight checksum mismatch.")
        temporary.replace(weight_path)
    actual_hash = hashlib.sha256(weight_path.read_bytes()).hexdigest()
    if actual_hash != WEIGHTS_SHA256:
        raise ValueError("Local pretrained weight checksum mismatch.")
    print("Official pretrained weights verified:", weight_path)
else:
    print("Demo mode: upstream ImageNet weights are not needed.")


## 2. Dataset access, checksum, and provenance

Use the official PlantSeg v7 archive from https://zenodo.org/records/17719108. The project-root ZIP matches its published MD5, and all 15,552 extracted files match the ZIP by size and CRC. The source extraction is read only; the audit writes a fixed manifest outside the dataset.

**本节做什么：**定位 `Metadata.csv`、图片和标注，保留官方的 train/val/test 划分。如果能访问原始 ZIP，就校验整个官方 v7 归档；如果只有解压数据，后面的单元会逐张核对本任务使用的图片。

In [ ]:
from plantseg_cw2.dataset import dataset_root, load_manifest, is_included, make_transforms, PlantDiseaseDataset
import zipfile

EXPECTED_ZIP_MD5 = "9358a66dff88cdd15c4fe009763c40a3"
ZENODO_ZIP_URL = "https://zenodo.org/records/17719108/files/plantseg.zip?download=1"
DOWNLOAD_DATA = False  # Set True only when the dataset has not been attached or extracted.
zip_candidates = [PROJECT_ROOT / "plantseg.zip", PROJECT_ROOT / "plantseg" / "plantseg.zip"]
archive = next((p for p in zip_candidates if p.is_file()), None)
try:
    DATA_DIR = dataset_root()
except FileNotFoundError:
    if archive is None:
        if not DOWNLOAD_DATA:
            raise FileNotFoundError(
                "PlantSeg extraction missing. Add the official v7 archive/extraction, set PLANTSEG_ROOT, "
                "or set DOWNLOAD_DATA=True to fetch the 1.1 GB archive."
            )
        archive = PROJECT_ROOT / "plantseg.zip"
        print("Downloading official PlantSeg v7 archive (about 1.1 GB)...")
        with urllib.request.urlopen(ZENODO_ZIP_URL, timeout=120) as remote, archive.open("wb") as local:
            shutil.copyfileobj(remote, local)
    digest = hashlib.md5()
    with archive.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    if digest.hexdigest() != EXPECTED_ZIP_MD5:
        raise ValueError("Dataset ZIP does not match the official PlantSeg v7 MD5.")
    destination = PROJECT_ROOT / "plantseg"
    destination.mkdir(exist_ok=True)
    resolved_destination = destination.resolve()
    with zipfile.ZipFile(archive) as bundle:
        for member in bundle.infolist():
            target = (destination / member.filename).resolve()
            if target != resolved_destination and resolved_destination not in target.parents:
                raise ValueError("Dataset ZIP contains a path outside the extraction directory.")
        bundle.extractall(destination)
    DATA_DIR = dataset_root()

if archive:
    digest = hashlib.md5()
    with archive.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    print("plantseg.zip MD5:", digest.hexdigest())
    if digest.hexdigest() != EXPECTED_ZIP_MD5:
        raise ValueError("Dataset ZIP does not match the official PlantSeg v7 checksum.")
else:
    print("Archive absent: extracted-file checks cannot independently verify v7 provenance.")
metadata_sha256 = hashlib.sha256((DATA_DIR / "Metadata.csv").read_bytes()).hexdigest()
print("Metadata.csv SHA-256:", metadata_sha256)
print("Extracted source:", DATA_DIR)


## 3. Fixed split and data audit

Metadata.csv supplies the host, disease, and original train/validation/test split. The audit checks readability, class counts, exact duplicates, and visually reviewed near-duplicate candidates. At the user's direction, every readable image remains in its supplied split: 4,349 images (3,007 train, 475 validation, 867 test) across 43 host-disease classes. No duplicate-based exclusions or split reassignment are made. Confirmed cross-split copies are documented, so validation and test scores may be optimistic for genuinely new photographs.

**输入 → 输出：**读取原始元数据和图片；生成 `manifest.csv`、`class_mapping.json`、`dataset_audit.json`。审计会检查坏图、类别数量及重复图片，但不因重复而移动或删除任何图片。

In [ ]:
from plantseg_cw2 import run_experiments

if MODE == "full":
    run_experiments.main(["--audit"])
elif not (RESULT_DIR / "manifest.csv").is_file():
    raise FileNotFoundError("Demo mode needs the saved manifest; run full mode first.")
audit = json.loads((RESULT_DIR / "dataset_audit.json").read_text(encoding="utf-8"))
rows = load_manifest(RESULT_DIR / "manifest.csv")
used_rows = [row for row in rows if is_included(row)]
class_map = json.loads((RESULT_DIR / "class_mapping.json").read_text(encoding="utf-8"))
class_names = [item["class_name"] for item in class_map["classes"]]
print("Audit:", json.dumps({key: audit.get(key) for key in
      ["raw_images", "used_images", "classes", "split_counts",
       "exact_duplicate_group_count", "cross_split_exact_duplicate_group_count",
       "cross_split_near_duplicate_candidate_count"]}, indent=2))
assert len(class_names) == len(set(class_names))
assert {row["split"] for row in rows} == {"train", "val", "test"}
assert len(used_rows) == len(rows) == audit['raw_images']
assert audit['duplicate_policy'] == 'original_splits_no_duplicate_exclusions'
# A source ZIP proves the complete archive version. Kaggle often exposes its extracted files
# without the uploaded ZIP. In that case, compare the metadata and every selected image
# with the hashes from our local copy of the officially verified v7 archive.
EXPECTED_METADATA_SHA256 = "03025e84fd4847e21a8cf1a7690b9c5ba40a933022f973df0a5611ffc0439fd2"
EXPECTED_TASK_IMAGES_SHA256 = "1c3f18c3447581adff49a628ba2c661ce7f5e419447043e059de96b59a6c2a11"
assert metadata_sha256 == EXPECTED_METADATA_SHA256, "Metadata.csv differs from the verified v7 copy"
fingerprint_lines = [f"{row['image_path']}\t{row['sha256']}" for row in
                     sorted(rows, key=lambda row: row['image_path'])]
task_images_sha256 = hashlib.sha256("\n".join(fingerprint_lines).encode()).hexdigest()
assert task_images_sha256 == EXPECTED_TASK_IMAGES_SHA256, "One or more task image hashes differ"
if archive is not None:
    assert audit['dataset_release'] == 'PlantSeg v7 verified'
    print("Complete v7 archive and extraction verified.")
else:
    print("No source ZIP: metadata and all 4,349 task images match the locally verified v7 copy;")
    print("the complete archive, including unused files, was not verified in this session.")
print('All original split images retained:', len(used_rows))

### 审计结果速读

下面按 train、val、test 汇总样本数，并检查 43 个类别是否都进入映射。若数量与预期不同，先核对数据集版本和目录，再运行训练。

In [ ]:
counts = Counter((row["split"], row["class_id"]) for row in used_rows)
for split in ("train", "val", "test"):
    subset = [row for row in used_rows if row["split"] == split]
    print(f"{split}: {len(subset)} images, {len({row['class_id'] for row in subset})} classes")
print("First five classes:")
for item in class_map["classes"][:5]:
    c = item["class_id"]
    print(item["class_name"], [counts[(split, c)] for split in ("train", "val", "test")])

### Distribution and representative training images

The class-frequency plot uses all images in the original supplied partitions. The example gallery uses training images only. All counts come from the audited manifest.

**如何阅读：**类别分布图显示每个类别在三份原始划分中的数量；示例图每种寄主取一张训练图片，仅供观察，不参与模型选型。

In [ ]:
from IPython.display import display, Image as NotebookImage

for name in ("class_distribution.png", "training_examples.png"):
    path = RESULT_DIR / "figures" / name
    if path.is_file():
        display(NotebookImage(filename=str(path)))
    else:
        print("Missing figure:", path)

## 4. Preprocessing and loaders

All models receive RGB images resized to 256 pixels and centre-cropped to 224 pixels. Validation and test transforms are deterministic. The augmentation experiment changes only the training transform by adding a horizontal flip, a small rotation, and mild colour jitter. Both use ImageNet V1 normalization for the pretrained ResNet-18. Class weights, when used, are computed from training rows only. No filenames or metadata are model inputs.

**预处理：**所有图片转 RGB，并按预训练权重要求缩放、中心裁剪、标准化。验证集和测试集不做随机增强，因此同一个模型的评估结果可重复。

### 数据加载检查

这一格实例化训练、验证和测试数据集，只显示数量和一条样本的形状；不会训练模型。

In [ ]:
train_plain = PlantDiseaseDataset(rows, DATA_DIR, "train", make_transforms(False))
val_set = PlantDiseaseDataset(rows, DATA_DIR, "val", make_transforms(False))
test_set = PlantDiseaseDataset(rows, DATA_DIR, "test", make_transforms(False))
x, y = train_plain[0]
print("Tensor shape:", tuple(x.shape), "label:", y, class_names[y])
print("Dataset lengths:", len(train_plain), len(val_set), len(test_set))
print("Training transform:", make_transforms(False))
print("Augmented transform:", make_transforms(True))

## 5. Majority-class baseline

The baseline predicts the most common class in the *training* partition for every input. It reads no image pixels. Its validation and final test metrics are produced by the shared evaluation code, using the same class order and undefined-metric convention as the CNN systems.

**为什么先做多数类基线：**它总预测训练集中数量最多的类别，不读取图片像素。后面的 CNN 应与这个简单参照比较，而不只看自己的准确率。

### 多数类基线结果

这里根据训练标签找到最多的类别，再在验证集上计算参照分数。

In [ ]:
train_labels = [row["class_id"] for row in used_rows if row["split"] == "train"]
val_labels = [row["class_id"] for row in used_rows if row["split"] == "val"]
majority_id = Counter(train_labels).most_common(1)[0][0]
majority_val_predictions = [majority_id] * len(val_labels)
print("Majority class:", class_names[majority_id])
print("Training support:", Counter(train_labels)[majority_id])
print("Validation prediction count:", len(majority_val_predictions))

## 6. CNN and controlled experiments

The basic system is an ImageNet-pretrained ResNet-18 with a new K-class linear head and frozen backbone. Three independent runs start from the same pretrained backbone and seed:

1. Basic CNN: deterministic training transforms and ordinary cross-entropy.
2. Augmentation: change only the training transform.
3. Class weighting: change only the loss weights, computed from training counts.

The script runs a one-batch forward/backward and save/load smoke test, records per-epoch loss, accuracy, validation macro-F1 and elapsed time, and saves each run's best checkpoint by validation macro-F1. The two changes are compared separately against the basic CNN. No test labels are used in this stage.

**实验设计：**三组 CNN 从相同的预训练骨干和随机种子开始。基础组只训练分类头；增强组只改训练图片增强；加权组只改损失函数的类别权重。这样差异更容易解释。

### 开始训练与冻结

`full` 模式会依次训练三组 CNN，并把验证集最优检查点保存在 `checkpoints/`。看到每个 epoch 的验证 macro-F1 属于正常输出；`demo` 和 `recorded` 模式读取既有结果。

In [ ]:
if MODE == "full":
    run_experiments.main(["--train"])
else:
    print("Demo mode: training skipped; loading saved experiment summaries.")
summary_path = RESULT_DIR / "experiment_summary.csv"
if not summary_path.is_file():
    raise FileNotFoundError(f"No experiment summary at {summary_path}")
with summary_path.open("r", encoding="utf-8", newline="") as stream:
    experiment_rows = list(csv.DictReader(stream))
for row in experiment_rows:
    print(row)

## 7. Validation comparison and checkpoint choice

The winner is selected solely by validation macro-F1. Histories and summary files below are generated from the actual runs. The training script freezes the experiment plan before the final test stage; rerunning after seeing test outcomes would invalidate the held-out protocol.

**如何选模型：**只用验证集的 macro-F1 选最佳检查点及最终模型。`experiment_summary.csv` 保存各组的验证分数和训练用时；测试集不参与这个选择。

### 训练曲线检查

每个实验的 CSV 历史记录包含训练损失、验证损失、验证 macro-F1 和用时。曲线用于解释训练过程，不用于改动已经冻结的测试方案。

In [ ]:
history_files = sorted((RESULT_DIR / "logs").glob("*history*.csv"))
print("History files:", [p.name for p in history_files])
for path in history_files:
    with path.open("r", encoding="utf-8", newline="") as stream:
        history = list(csv.DictReader(stream))
    print(path.name, "epochs:", len(history), "last row:", history[-1] if history else None)
for path in sorted(RESULT_DIR.glob("*selection*.json")) + sorted(RESULT_DIR.glob("*plan*.json")):
    print(path.name, path.read_text(encoding="utf-8")[:3000])

### 验证集比较

这一格把基础模型与增强、类别加权模型放在同一指标下比较。分数是验证集结果，不是最终测试分数。

In [ ]:
validation_by_name = {row["experiment"]: float(row["val_macro_f1"]) for row in experiment_rows}
if "baseline_resnet18" in validation_by_name:
    base_f1 = validation_by_name["baseline_resnet18"]
    for name in ("augmentation_resnet18", "weighted_resnet18"):
        if name in validation_by_name:
            change = validation_by_name[name] - base_f1
            direction = "improved" if change > 0 else "reduced" if change < 0 else "matched"
            print(f"{name} {direction} validation macro-F1 by {abs(change):.4f} versus the basic CNN.")
selected_validation = max(experiment_rows, key=lambda row: float(row["val_macro_f1"]))
print("Selected by validation macro-F1:", selected_validation["experiment"],
      f"({float(selected_validation['val_macro_f1']):.4f})")


## 8. Final test evaluation

The final test command is a separate, explicit phase. It evaluates the frozen majority baseline and every CNN system on the official test rows, using the same deterministic transform, fixed class order, model.eval(), and disabled gradients. The command also produces the full confusion matrix and per-class precision, recall, F1 and support. If this notebook is rerun, inspect the saved final-test outputs before repeating the stage; do not use test scores to revise training choices.

**测试集的角色：**模型选择写入 `frozen_plan.json` 后，再对测试集做最终评估。第一次运行会写 `final_test_metrics.json`；后续重复运行会复用同一冻结计划的结果。

### 执行最终测试

只有冻结计划就绪后，`full` 模式才评估测试集。若运行中断，检查 `frozen_plan.json` 和检查点是否完整，再继续。

In [ ]:
if MODE == "full":
    run_experiments.main(["--final-test"])
else:
    print("Demo mode: final test skipped; using saved outputs.")
if (RESULT_DIR / "experiment_summary.csv").is_file():
    with (RESULT_DIR / "experiment_summary.csv").open("r", encoding="utf-8", newline="") as stream:
        for row in csv.DictReader(stream):
            print(row)

### 读取最终测试分数

这里展示已保存的测试指标以及选中的模型。报告中应区分 validation macro-F1 和 test macro-F1。

In [ ]:
final_path = RESULT_DIR / "final_test_metrics.json"
if final_path.is_file():
    final_results = json.loads(final_path.read_text(encoding="utf-8"))
    selected_name = final_results["selected_model"]
    for name, metrics in final_results["experiments"].items():
        print(f"{name}: test macro-F1={metrics['macro_f1']:.4f}, "
              f"accuracy={metrics['accuracy']:.4f}, "
              f"balanced accuracy={metrics['balanced_accuracy']:.4f}")
    print("Selected model on test:", selected_name)
    experiment_metrics = final_results["experiments"]
    if "baseline_resnet18" in experiment_metrics and "weighted_resnet18" in experiment_metrics:
        training_support = Counter(row["class_id"] for row in used_rows if row["split"] == "train")
        rare_ids = sorted(range(len(class_names)), key=lambda i: (training_support[i], i))[:8]
        basic = experiment_metrics["baseline_resnet18"]["per_class"]
        weighted = experiment_metrics["weighted_resnet18"]["per_class"]
        print("Eight lowest-support training classes: basic vs weighted test recall / F1")
        for class_id in rare_ids:
            b, w = basic[class_id], weighted[class_id]
            print(f"{class_names[class_id]} (train n={training_support[class_id]}): "
                  f"recall {b['recall']:.3f} -> {w['recall']:.3f}; "
                  f"F1 {b['f1']:.3f} -> {w['f1']:.3f}")
    error_path = RESULT_DIR / "error_analysis.json"
    if error_path.is_file():
        errors = json.loads(error_path.read_text(encoding="utf-8"))
        print("Frequent observed confusions:")
        for item in errors.get("top_confusions", [])[:5]:
            print(item)
else:
    print("Final test metrics are not available in this workspace.")


### Post-test duplicate-overlap sensitivity check

This is an exploratory analysis of the already-frozen selected model. It removes test rows with an audited exact or visually confirmed same-photo match in training from the **analysis only**. The original split, training, checkpoint choice, and official test metrics are unchanged. Undetected overlaps and validation overlap remain possible.

**结果解读：**这里用已保存的测试预测，查看去掉“检测到与训练图片相同照片”的测试样本后分数如何变化。这是结果局限性分析，不会重新选模型。

### 重复照片敏感性分析

这一步读取审计记录、人工复核记录和已保存预测，报告完整测试集与“未检测到训练照片重复”子集的分数。该子集不是一个新的官方测试集。

In [ ]:
from plantseg_cw2.analysis import sensitivity_analysis
sensitivity = sensitivity_analysis.main(RESULT_DIR)
print("Official test:", sensitivity["all_test"])
print("Without detected training-photo overlap:", sensitivity["without_detected_train_overlap"])


## 9. Regenerate the report's figures and tables

The plotting stage reads the fixed audit, per-epoch histories, saved predictions and evaluation tables. It creates every data-derived figure and result table used by the report. It can be repeated without retraining. The complete confusion matrix stays available below even if the report displays a readable subset.

**输出位置：**图表保存在 `RESULT_DIR/figures/`，预测表和混淆矩阵保存在 `RESULT_DIR/tables/`。这些图表读取已生成的结果，不会重新训练。

### 重画结果图

使用现有审计和预测文件重新生成类别分布、训练曲线和混淆矩阵等图片。图片会写入结果目录。

In [ ]:
run_experiments.main(["--plots"])
figure_dir = RESULT_DIR / "figures"
report_figures = ("class_distribution.png", "training_examples.png", "training_curves.png",
                  "model_comparison.png", "confusion_matrix.png", "prediction_examples.png")
for filename in report_figures:
    path = figure_dir / filename
    if path.is_file():
        print(path.name)
        display(NotebookImage(filename=str(path)))


### 混淆矩阵

行表示真实类别，列表示预测类别；对角线越大表示该类预测正确的数量越多。请结合每类样本数解释。

In [ ]:
matrix_path = RESULT_DIR / "tables" / "confusion_matrix.csv"
if matrix_path.is_file():
    with matrix_path.open("r", encoding="utf-8", newline="") as stream:
        matrix_rows = list(csv.reader(stream))
    print("Full confusion matrix:", len(matrix_rows) - 1, "rows")
    for row in matrix_rows:
        print(", ".join(row))
else:
    print("Run the final-test stage to create the full confusion matrix.")

for path in sorted((RESULT_DIR / "tables").glob("*per_class*.csv")):
    print("\nPer-class report:", path.name)
    with path.open("r", encoding="utf-8", newline="") as stream:
        for row in csv.DictReader(stream):
            print(row)

## 10. Prediction examples and error analysis

Inspect the saved prediction gallery and tables for concrete successes and failures. Possible mechanisms include similar symptoms, small lesions, blur, backgrounds, and class imbalance. Such explanations are hypotheses unless the pictured errors support them. Maximum softmax score is shown as a model score, not a calibrated probability. This section intentionally makes no score or error claim before the actual test outputs exist.

**阅读提示：**先看预测表中的真实类别、预测类别和置信分数，再看最常见的混淆对。置信分数是 softmax 分数，不能直接当成校准后的真实概率。

### 查看预测与错误样本

这里只读取表格，帮助挑选报告或展示中能解释模型成功与失败的具体样本。

In [ ]:
for path in sorted((RESULT_DIR / "tables").glob("*prediction*.csv")):
    print("Prediction table:", path.name)
    with path.open("r", encoding="utf-8", newline="") as stream:
        data = list(csv.DictReader(stream))
    mistakes = [row for row in data if row.get("true_class") != row.get("pred_class")]
    print("Rows:", len(data), "candidate mistakes:", len(mistakes))
    for row in mistakes[:10]:
        print(row)

## 11. Inference demonstration

The command below loads the saved selected checkpoint and class mapping, preprocesses one RGB image, and prints the predicted host–disease class and maximum softmax score. It uses a validation image only as a readily available example; replace DEMO_IMAGE with a new image path for the video. The inference code never reads ground-truth labels as inputs.

**现场演示：**加载验证集表现最好的检查点，对一张验证图片给出前几个预测类别。运行这一格必须已有模型检查点；只有保存的指标文件不足以做新图片推理。

### 推理演示

选择原始验证集的一张图片运行模型。`recorded` 模式只有历史指标时会提示需要重新生成检查点。

In [ ]:
if MODE == "recorded":
    print("Saved results are available, but this Git checkout has no model weights. "
          "Run full mode to regenerate checkpoints before live inference.")
else:
    from plantseg_cw2.inference import predict
    demo_row = next(row for row in used_rows if row["split"] == "val")
    DEMO_IMAGE = DATA_DIR / demo_row["image_path"]
    print("Demo image:", DEMO_IMAGE)
    print(json.dumps({"image": str(DEMO_IMAGE),
                      "predictions": predict(DEMO_IMAGE)}, indent=2))
    print("Softmax scores are not calibrated probabilities.")


## 12. Reproducibility and references

The result directory records the manifest, class mapping, dataset audit, environment, seed, configurations, measured training times, histories, checkpoints, test predictions, metrics and figures. The archive checksum verifies a downloaded ZIP when present; the extracted files in this local workspace alone do not prove their release version. Re-running a full experiment can vary slightly across CUDA kernels, and any such variation must be described against the saved result.

**References:** Wei et al. (2025), PlantSeg v7 dataset, https://doi.org/10.5281/zenodo.17719108; Wei et al. (2026), *Scientific Data* 13, 205, https://doi.org/10.1038/s41597-025-06513-4; Chilamkurthy (2025), PyTorch transfer learning tutorial, https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html.

**Submission check:** Fill in student identity and team contributions in the report and notebook. Record the actual video separately; the script is a recording guide, not a submitted video.

**复现实验：**`result/` 包含数据清单、环境版本、训练配置、检查点、预测和图表。不同 GPU 或软件版本可能产生小幅数值差异，报告应以实际运行输出为准。

## 13. Supplementary fine-tuning extension

This section was added after the original final test was already visible. It therefore has a separate frozen extension plan and a **supplementary**, not blind, test result. The original PlantSeg split and all 4,349 nine-host images remain unchanged. The model fine-tunes ResNet-18 layer4 and its head, selects its checkpoint by validation macro-F1, and stops a continuation after five epochs without improvement. In demo mode, the saved records below are loaded without retraining.

**重要时间顺序：**下面的微调是原始最终测试结果已经可见后增加的补充实验。它单独记录验证、冻结和测试，不应把它的测试分数描述成完全盲测的原始主结果。

### 补充微调的执行顺序

先从基础模型检查点微调并仅用验证集选最佳权重；随后冻结补充计划、评估一次测试集、比较两套系统。该部分耗时较长，且不能用于反过来修改原始模型选择。

In [ ]:
from plantseg_cw2.analysis import explore_finetune, validation_overlap_analysis, evaluate_extension, compare_extension
if MODE == "full":
    explore_finetune.main(epochs=10, patience=4)
    explore_finetune.main(epochs=15, patience=5, continue_from_best_pilot=True)
    validation_overlap_analysis.main()
    evaluate_extension.freeze(RESULT_DIR)
    evaluate_extension.evaluate(RESULT_DIR)
    compare_extension.main()
else:
    print("Demo mode: using saved supplementary experiment records.")
for name in ("summary.json", "frozen_extension_plan.json", "comparison.json"):
    path = RESULT_DIR / "exploratory_finetune" / name
    if path.is_file():
        record = json.loads(path.read_text(encoding="utf-8"))
        print(name, {key: record[key] for key in record if key in
                     ("best_epoch", "best_validation", "validation_macro_f1", "metrics", "label")})
validation_path = RESULT_DIR / "validation_overlap_analysis.json"
if validation_path.is_file():
    record = json.loads(validation_path.read_text(encoding="utf-8"))
    print("Full/overlap-free validation macro-F1:")
    for name, item in record["experiments"].items():
        print(name, item["all_validation"]["macro_f1"],
              item["without_detected_train_overlap"]["macro_f1"])
for filename in ("finetune_curves.png", "extension_comparison.png"):
    path = RESULT_DIR / "figures" / filename
    if path.is_file():
        display(NotebookImage(filename=str(path)))
